# Fixify → Documentación del front-end, versión 2

**Fecha: 02-Oct-2026.** Esta versión documenta el front-end refactorizado, con especial atención a `services/api_client.py`, httpx y asyncio. La versión 1 se conserva como referencia histórica.

La aplicación es de escritorio: Tkinter/ttkbootstrap dibuja ventanas; FastAPI recibe peticiones; PostgreSQL en Supabase guarda datos. El frontend no se conecta directamente a la BD.

## Cómo leer y ejecutar

1. Para entender la API, leer las secciones 1–9 en orden.
2. Para entender ventanas y asincronía, continuar con las secciones 10–13.
3. Para comprobar el comportamiento, ejecutar los ejemplos simulados de la sección 14.
4. Para localizar una clase/método, consultar el inventario y el código de referencia de la sección 15.

Las celdas ejecutables usan exclusivamente `httpx.MockTransport`: **no envían tráfico de red, no abren ventanas ni leen credenciales**. Los fragmentos de la aplicación están en Markdown, por lo que Run All no ejecuta la GUI. Seleccionar un kernel con las dependencias de `frontend/requirements.txt`. No se necesita instalar un paquete llamado asyncio: pertenece a Python.

La revisión y pruebas locales no acreditan conectividad real con Supabase ni aceptación visual completa.

## 1. Qué responsabilidad tiene cada pieza

```text
Ventana / vista (Tk, hilo principal)
  lee campos y pide una operación
       |
       |
AsyncRunner.submit(coroutine, owner, on_success, on_error)
       → agenda trabajo en el loop de un hilo separado
       |
Service de dominio (Auth / Clients / Equipment / Users)
  decide ruta, verbo y datos
       → await
       |
ApiClient (nuestro adaptador común)
  agrega JWT, administra HTTP y transforma errores
       → await
       |
httpx.AsyncClient (objeto de la biblioteca httpx)
  transporta la solicitud y obtiene una Response
       |
       |
FastAPI → repositorio SQLAlchemy → PostgreSQL/Supabase
       → respuesta
       |
cola Python → root.after → callback de la vista (hilo Tk)
```

`ApiClient` y `httpx.AsyncClient` son clases diferentes. La primera es código de Fixify; la segunda es la biblioteca que realiza HTTP. `ModuleService` comparte transporte y validación de listas; cada service hereda de esa base. `Services` construye las instancias y las entrega a las vistas: no realiza solicitudes por sí mismo.

Se comparten objetos, no se copian: `services.clients.api is services.api` y `services.equipment.api is services.api` son verdaderos. Cambiar el token del transporte afecta a las próximas solicitudes de todos los módulos.

## 2. Vocabulario mínimo de HTTP y Python

| Concepto | Significado en Fixify | Ejemplo |
|---|---|---|
| URL base | Dirección del backend | `http://127.0.0.1:8000` |
| Ruta | Recurso relativo | `/equipos/` |
| GET | Consulta sin alta | Listar equipos |
| POST | Envío para crear o autenticar | Registrar equipo / login |
| PATCH | Cambio parcial | Cambiar rol |
| Headers | Metadatos HTTP | Authorization |
| Body | Datos del envío | JSON con marca/modelo |
| Query params | Filtros en URL | `/clientes/buscar?q=Ana` |
| Response | Objeto recibido de httpx | Status, headers y contenido |
| `dict` / `list` | Datos Python después de decodificar JSON | Una entidad / varias entidades |
| `self` | La instancia actual de una clase | `self.token` pertenece al transporte |
| `_nombre` | Convención de miembro interno | `_client` no es un service de dominio |
| Coroutine | Operación async pendiente | `service.listar()` |
| Callback | Función que se llama al terminar | `_loaded(records)` |

Una petición es un viaje al servidor. Una coroutine es una operación pendiente de Python. Crear una coroutine no ejecuta todavía ese viaje. El callback es la función que procesa su resultado; no es la respuesta HTTP.

## 3. ApiClient: estado, constructor y ciclo de vida

Archivo: `frontend/src/services/api_client.py`.

| Atributo | Valor inicial | Uso |
|---|---|---|
| `base_url` | Argumento, variable FIXIFY_API_URL o dirección local | Localizar backend |
| `token` | None | JWT compartido de la sesión |
| `_transport` | Argumento opcional | Transporte real por defecto o simulador de pruebas |
| `_client` | None | AsyncClient creado en la primera solicitud |

`base_url or os.getenv(...)` elige una URL explícita si es verdadera; si se omite o está vacía, consulta el entorno. La URL no contiene las credenciales de Supabase. El argumento `transport` permite conectar el mismo código a un simulador; no representa una tabla de BD.

El constructor solo guarda configuración. La primera llamada a `request` crea `_client`; las siguientes lo reutilizan. Se evita abrir un cliente por cada registro o cada clic. Su creación y uso ocurren en el loop del trabajador, el mismo donde se cierran las conexiones.

```text
ApiClient creado → _client=None
primera request → se crea AsyncClient
siguientes request → mismo AsyncClient
aclose → se cierran conexiones → _client=None
```

`set_token(token)` no llama al servidor: cambia memoria local. `set_token(None)` elimina la credencial de futuras peticiones, pero no revoca un JWT ya emitido en el backend.

## 4. La firma de request, argumento por argumento

```python
async def request(self, method, path, *, authenticated=True, **kwargs):
```

| Elemento | Qué significa | Cómo se utiliza |
|---|---|---|
| `async def` | Define una coroutine | Se espera con await o se agenda en AsyncRunner |
| `self` | ApiClient compartido | Da acceso a URL, token y conexiones |
| `method` | Verbo HTTP | GET, POST, PATCH; httpx también admite otros |
| `path` | Ruta relativa | `/clientes/` |
| `*` | A partir de aquí los argumentos son por nombre | `authenticated=False` |
| `authenticated=True` | Habilita agregar JWT si hay uno | Login lo desactiva |
| `**kwargs` | Reúne argumentos adicionales en un diccionario | json, data, params |

Ejemplo de expansión:

```python
await api.request('POST', '/clientes/', json={'nombre': 'Ana', 'telefono': '3312345678'})
```

Dentro de `request`, `kwargs` contiene `{'json': {...}}`. Luego `**kwargs` en la llamada a httpx vuelve a expandirlo como argumentos por nombre. No se modifica el payload por magia: el service lo construye explícitamente.

`authenticated=True` **no valida** una cuenta ni garantiza que exista token. Si token es None se envían headers vacíos y el servidor decide si rechaza. No se deben pasar `headers` dentro de kwargs con el diseño actual: `request` ya pasa ese argumento, y repetirlo causaría TypeError. Cambios futuros de headers deben centralizarse en el transporte.

## 5. request por etapas: desde el clic hasta los datos

### 5.1 Preparar el cliente

`if self._client is None` crea `httpx.AsyncClient(base_url=..., timeout=5, transport=..., follow_redirects=True)`. El proyecto configura un timeout de cinco segundos y sigue redirecciones. Ese timeout controla esperas de operaciones HTTP; no debe interpretarse como un cronómetro total exacto para todo el flujo de UI. [Referencia de timeout HTTPX](https://www.python-httpx.org/advanced/timeouts/).

### 5.2 Preparar autorización

```python
headers = {'Authorization': f'Bearer {self.token}'} if authenticated and self.token else {}
```

Solo se agrega la cabecera cuando ambos valores lo permiten. Bearer anuncia al backend el esquema de autenticación. No imprimir token, contraseña ni cabeceras reales en logs/notebooks.

### 5.3 Esperar la red

```python
response = await self._client.request(method, path, headers=headers, **kwargs)
```

Mientras la coroutine espera una operación de E/S, el loop puede atender otras tareas. En Fixify ese loop está en el hilo de trabajo; Tk sigue procesando eventos en su hilo. El archivo no mueve widgets ni llama a messagebox.

### 5.4 Interpretar respuesta

`response.is_error` identifica errores HTTP y hace que el adaptador lance ApiError. Si la respuesta es 204 se devuelve None: no hay contenido para decodificar. En otro caso `response.json()` retorna datos Python. La aplicación no devuelve el objeto Response a las vistas, sino el contenido decodificado.

Que la respuesta sea 200 no garantiza que su JSON tenga el contrato correcto. `ModuleService._list` comprueba lista y valida cada registro con su modelo; `AuthService.login` comprueba token y rol. Cada entidad se valida mediante un contrato Pydantic antes de entregarla a la vista.

## 6. JSON, formulario y filtros: no son intercambiables

| Argumento | Uso actual | Ejemplo del módulo |
|---|---|---|
| `json={...}` | Cuerpo JSON | Alta cliente/equipo/usuario y cambio de rol |
| `data={...}` | Formulario codificado | POST /auth/login |
| `params={...}` | Filtros en la URL | Buscar cliente por q |

`AuthService.login` usa formulario porque el backend recibe OAuth2PasswordRequestForm. Si se cambia a JSON sin cambiar el backend, el contrato no coincide. `ClientsService.buscar` deja a httpx codificar el término: no concatena texto manualmente en la URL. Los opcionales de alta se convierten de cadena vacía a None, que se serializa como null.

Ejemplos del código local:

```python
await api.request('POST', '/auth/login', authenticated=False,
                  data={'username': username, 'password': password})
await api.request('POST', '/equipos/', json=payload)
await api.request('GET', '/clientes/buscar', params={'q': termino})
```

La configuración de parámetros y cuerpos se apoya en la [guía de HTTPX](https://www.python-httpx.org/quickstart/); las rutas y conversiones anteriores provienen de los services de Fixify.

## 7. ApiError: errores de transporte, HTTP y contrato

`ApiError` hereda de Exception, conserva un mensaje legible y un `status_code` opcional. `super().__init__(message)` permite que `str(error)` sea el mensaje mostrado por la UI.

| Situación | Tratamiento local | status_code |
|---|---|---|
| TimeoutException | «El servidor tardó demasiado.» | None |
| RequestError | «No se pudo conectar.» | None |
| HTTP 401/403/404/422/500 | `_error_message` extrae detail o genera mensaje HTTP | Código recibido |
| Respuesta no JSON cuando se espera JSON | «Respuesta inválida» | None |
| Lista o login con estructura incorrecta | El service/base lanza ApiError | None |

Timeout se captura antes de RequestError para conservar un mensaje específico. `raise ... from exc` mantiene la causa para depuración, mientras la UI utiliza el texto simple del nuevo error.

`_error_message` es staticmethod: no utiliza token ni estado de la instancia, solo la respuesta recibida. Si detail es lista de errores Pydantic, une sus mensajes con saltos de línea; si es cadena, la conserva. Si no entiende la respuesta, genera `La solicitud falló (HTTP N).` No existe reintento automático.

Cancelación no se interpreta como éxito ni como fallo HTTP. AsyncRunner descarta futures cancelados. Si el servidor ya confirmó una escritura, cancelar la espera no la deshace: antes de repetir un alta, consultar si se guardó.

## 8. ModuleService, herencia e inyección

```python
class EquipmentService(ModuleService):
    async def listar(self):
        return await self._list('/equipos/', Equipo)
```

EquipmentService recibe el constructor de ModuleService por herencia. `ModuleService.__init__(api)` conserva `self.api`; `_list(path, **kwargs)` hace GET, recibe JSON, exige una lista de diccionarios y convierte cada elemento al modelo indicado. Un listado vacío `[]` es válido.

`Services(api=None)` usa el ApiClient recibido o crea uno. Construye cuatro services que apuntan al mismo transporte. Las vistas reciben `services` y `runner` en sus constructores. Esa inyección permite sustituir solo el transporte en pruebas y conservar los services reales.

| Clase / archivo | Operaciones | Transformación de entrada |
|---|---|---|
| AuthService / auth_service.py | login | Formulario; valida respuesta; no guarda JWT |
| ClientsService / clients_service.py | listar, crear, buscar | Opcionales a None; q en params |
| EquipmentService / equipment_service.py | listar, crear | client_id a int; serie vacía a None |
| UsersService / users_service.py | listar, crear, cambiar_rol, activar | PATCH parcial; elige habilitar/inhabilitar |

Los services no contienen controles de Tk, no deciden colores, no leen Entry y no abren ventanas. Tampoco son controles de seguridad suficientes: el backend vuelve a autorizar cada operación.

## 9. Recorrido de equipos y relación con el diagrama de BD

### Consultar

1. CollectionView.load pide la coroutine EquipmentView.fetch.
2. EquipmentView llama EquipmentService.listar.
3. ModuleService._list llama ApiClient.request GET `/equipos/`.
4. httpx obtiene el JSON del router; se valida lista y se obtiene una lista de modelos Equipo.
5. AsyncRunner entrega la lista a CollectionView._loaded en Tk.
6. EquipmentView.row transforma cada registro a columnas de tabla.

### Registrar

1. El botón Nuevo consulta clientes sin bloquear la UI.
2. La vista crea un mapa etiqueta visible → ID y un formulario.
3. El usuario completa campos; FormDialog lee valores en el hilo Tk.
4. La coroutine de guardado convierte la etiqueta de propietario al ID.
5. EquipmentService construye payload JSON y llama POST `/equipos/`.
6. El backend comprueba cliente, inserta y responde 201 con folio/detalles.
7. El callback informa alta y vuelve a cargar el listado.

El backend sigue router → repositorio → modelo/esquema. La implementación de equipos ahora reconoce los nombres del diagrama entregado:

| API / atributo Python | Columna física SQL |
|---|---|
| id | id_equipo |
| client_id | id_cliente |
| problema_reportado | falla_reportada |
| tipo, marca, modelo, numero_serie | Mismo nombre |

La FK apunta a `clientes.id_cliente`. El GET incluye numero_serie, incluso null cuando no existe. El orden del listado es ascendente por ID. Tanto POST como GET exigen admin/recepcion. `EQ-001` identifica equipo, no una orden de servicio.

El cambio es de mapeo ORM y endpoint; no ejecuta renombrados ni migraciones en Supabase. El diagrama no sustituye inspeccionar el esquema remoto. Otros módulos conservan diferencias previas, por ejemplo la columna direccion de clientes y el diseño users frente a usuarios/roles.

## 10. async, await y el puente con Tk

`async def listar` produce una coroutine al llamarse. `await` espera su resultado dentro de otra coroutine. Tk espera funciones normales en command; por ello un botón no debe recibir directamente una función async.

```python
# Callback normal de Tk:
self.runner.submit(self.services.clients.listar(), self, self._loaded, self._failed)
```

No se usa `asyncio.run` dentro de un callback: ese método esperaría allí a que termine el flujo. En notebook se puede usar await directamente. En un script independiente sin GUI, asyncio.run puede ser adecuado como entrada única.

### AsyncRunner por método

| Método | Trabajo |
|---|---|
| __init__(root) | Crea loop, cola, registro de futures y trabajador; programa after |
| _run() | Instala el loop en el hilo y lo mantiene activo; cierra async generators al salir |
| submit(coroutine, owner, on_success, on_error) | Agenda coroutine y registra dueño/callbacks |
| cancel_owner(owner) | Elimina callbacks y solicita cancelación de las tareas de ese dueño |
| _poll() | En el hilo Tk, toma futures terminados de la cola y llama al callback correspondiente |
| close(cleanup, on_closed) | Inicia cancelación y cierre HTTP sin bloquear Tk |
| _wait_thread(on_closed) | Espera con after hasta que el hilo termina y cierra la ventana |

`run_coroutine_threadsafe` devuelve un Future de concurrent.futures para coordinar hilos. Su done callback solo pone el Future en Queue; no llama a Tk. `_poll` obtiene result() después de finalizar el Future, comprueba que la vista exista y procesa resultado/error. El polling ocurre aproximadamente cada 30 ms; no significa que haya una llamada de red cada 30 ms. [Referencia de asyncio](https://docs.python.org/3/library/asyncio-task.html#asyncio.run_coroutine_threadsafe).

## 11. Concurrencia, cancelación y cierre

El dashboard realiza consultas independientes con asyncio.gather y return_exceptions=True. Los resultados conservan el orden de las consultas; un fallo se representa como excepción en esa posición y no pierde las métricas que sí cargaron. Usuarios solo se consulta para admin; técnico no intenta leer endpoints prohibidos de clientes/equipos.

Cerrar una vista cancela las tareas de su owner. Cerrar el formulario hace lo mismo para su guardado. CollectionView.load cancela su lectura anterior antes de pedir una nueva; así reduce recargas desactualizadas. FormDialog conserva una ventana por módulo, y botones deshabilitados evitan repetir clics durante la espera.

`AsyncRunner.close` deja de programar polling, cancela tareas, espera cancelaciones, ejecuta `api.aclose()` en el loop de trabajo, detiene el loop y comprueba que el hilo terminó antes de destruir la ventana principal. `aclose` libera HTTP y deja `_client=None`. [Referencia del ciclo de vida AsyncClient](https://www.python-httpx.org/async/).

No todas las esperas son cancelación de negocio: el backend puede haber guardado antes de recibir la desconexión. No hay WebSocket, suscripción Supabase ni actualización automática desde eventos remotos: la reactividad actual mantiene la UI disponible durante HTTP y actualiza datos cuando llegan. Otros cambios externos requieren Actualizar o nueva consulta.

## 12. Componentes independientes y vistas

El antiguo `ui/components.py` se reemplazó por el paquete `ui/components/`. Cada archivo tiene una responsabilidad y conserva comentarios breves sobre sus argumentos y comportamiento.

| Archivo | Componente | Responsabilidad y argumentos |
|---|---|---|
| `field.py` | `Field` | Dataclass inmutable: name es la clave API, label el texto, required indica obligatoriedad, secret oculta la entrada y choices define opciones. |
| `async_frame.py` | `AsyncFrame` | Base de vistas: recibe master, services, runner y role. Al destruirse cancela las tareas de su propietario. |
| `table.py` | `create_table` | Construye Treeview con scroll vertical; parent es el contenedor, columns los encabezados y height las filas visibles. Devuelve la tabla. |
| `form_dialog.py` | `FormDialog` | Modal con campos, validación y guardado async. Recibe master, title, fields, runner, save y on_saved. |
| `collection_view.py` | `CollectionView` | Comparte listado, recarga, alta y estados. Las subclases configuran title, columns, fields e implementan fetch, save y row. |
| `__init__.py` | Exportaciones | Permite conservar `from ui.components import Field, CollectionView` en las vistas. |

### Dependencias y uso

`CollectionView` utiliza `AsyncFrame`, `FormDialog` y `create_table`. Los imports internos son relativos; las vistas consumen las exportaciones del paquete. Separar archivos permite modificar una pieza sin concentrar todos los controles en un solo archivo; los componentes conservan las dependencias necesarias para colaborar.

```python
from ui.components import CollectionView, Field
# También puede importarse directamente la pieza:
from ui.components.form_dialog import FormDialog
```

`FormDialog` lee controles en el hilo Tk, elimina espacios de datos normales y conserva los de contraseñas. `save(values)` produce la coroutine enviada al runner; `on_saved(result)` recibe la confirmación. El backend sigue siendo responsable de validar formatos e integridad.

### Botón Actualizar y asyncio

Al abrir una colección y después de un alta se llama a `load()`. Actualizar permite repetir la consulta para ver cambios de otros usuarios o reintentar un error. Durante la carga se deshabilita el botón; al recibir datos o un error se habilita nuevamente.

`asyncio` evita bloquear la interfaz durante las solicitudes; no detecta cambios remotos. No existe polling ni una suscripción en tiempo real. La recarga usa el mismo service y cliente HTTP compartido, sin crear otro bucle de eventos.


Actualización: AsyncFrame.request encapsula submit y el propietario; CollectionView usa service_name para listar/crear por defecto. Las vistas declaran menu_label, allowed_roles y menu_order para descubrimiento automático. Ver el tutorial actualizado para nuevas integraciones.

## 13. Sesión, navegación y mantenimiento

`app/main.py` prepara imports desde src y crea VentanaLogin. Login lee credenciales, deshabilita botón, muestra Conectando y envía trabajo al runner. AuthService valida respuesta, pero **no asigna token**: el callback de login lo hace si sigue activo. Luego abre SistemaGestion y oculta login.

SistemaGestion recibe rol de dominio, no etiqueta traducida, y decide menús. Reemplazar vista destruye controles y tareas anteriores. Logout limpia JWT local, destruye gestión y vuelve a login. Cerrar aplicación usa el cierre coordinado de recursos. Recuperación de cuenta informa que se contacte al administrador: no existe envío automático.

### Cómo agregar o cambiar una operación

- Cambiar ruta/payload en el service del dominio; conservar manejo HTTP en ApiClient.
- Para nuevos filtros usar params; para cuerpos usar el contrato acordado json/data.
- Si cambia la estructura de salida, ajustar esquema backend, validación y row de la vista.
- Para un módulo de listado/alta, crear una subclase de CollectionView y declarar sus metadatos; el menú la descubre sin registro manual.
- Actualizar pruebas de contratos y esta documentación cuando cambie comportamiento.

`models/database.py` conserva un prototipo de usuarios en memoria; no participa en login actual. No se reproducen sus contraseñas en este notebook.

### Límite de esta implementación

No hay refresh/revocación de JWT, política de retries, cache o paginación. Los permisos se basan también en backend y sus dependencias actuales mantienen rol del JWT emitido. Estas funciones no se atribuyen a httpx por el hecho de usar AsyncClient.

### Logo compartido

`ui/branding.py` centraliza la carga de `assets/img/Logo Fixify.png`. La ruta se resuelve desde el archivo del módulo, independientemente del directorio desde el que se inicia la app.

- `load_logo(master, size)` abre el PNG con Pillow, conserva transparencia RGBA y ajusta el tamaño sin deformarlo con thumbnail/LANCZOS. Devuelve una PhotoImage asociada al widget Tk indicado.
- `set_window_icon(window)` asigna el icono de la ventana y conserva la referencia en `_fixify_icon`.
- Login muestra el logo de hasta 140 x 140 px y utiliza una ventana de 450 x 540 px. El menú lateral muestra el logo de hasta 150 x 150 px. Ambas ventanas tienen icono.

Las ventanas guardan `self.logo`: Tk necesita que la referencia Python siga viva mientras se muestra la imagen. La carga ocurre en el hilo Tk, no mediante AsyncRunner. Pillow ya forma parte del entorno usado por ttkbootstrap.


## 14. Laboratorio seguro: httpx con respuestas simuladas

Las siguientes celdas importan **los services reales** y usan MockTransport. Los datos son sintéticos; el token de demostración nunca se imprime. Ejecutar celdas en orden desde el repositorio o su carpeta docs. Cada ejemplo cierra su cliente en finally.

El handler simulado hace el papel del servidor y permite inspeccionar JSON, formulario y headers sin una conexión externa.

In [1]:
from pathlib import Path
import sys
import asyncio
import json
import httpx

raiz = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / 'frontend/src/services/api_client.py').is_file()), None)
if raiz is None:
    raise RuntimeError('Abra el notebook dentro del repositorio Fixify.')
sys.path.insert(0, str(raiz / 'frontend/src'))
from services import Services
from services.api_client import ApiClient, ApiError
print('Services importados. No se abrió GUI ni conexión de red.')

Services importados. No se abri? GUI ni conexi?n de red.


In [2]:
async def ejemplo_alta_y_listado():
    registros = []
    solicitudes = []

    def servidor(request):
        # Observamos presencia de JWT, no su contenido.
        solicitudes.append((request.method, request.url.path,
                             'authorization' in request.headers))
        if request.url.path == '/auth/login':
            assert request.headers['content-type'].startswith('application/x-www-form-urlencoded')
            return httpx.Response(200, json={'access_token': 'token-sintetico', 'token_type': 'bearer', 'role': 'recepcion'})
        if request.method == 'POST':
            entidad = {'id': len(registros) + 1, **json.loads(request.content)}
            registros.append(entidad)
            return httpx.Response(201, json={'folio': f"EQ-{entidad['id']:03d}", 'mensaje': 'Guardado', 'detalles': {
                'id_interno': entidad['id'], 'tipo': entidad['tipo'], 'marca': entidad['marca'],
                'modelo': entidad['modelo'], 'cliente_id': entidad['client_id']}})
        return httpx.Response(200, json=registros)

    api = ApiClient(transport=httpx.MockTransport(servidor))
    services = Services(api)
    try:
        sesion = await services.auth.login('demo', 'clave-sintetica')
        api.set_token(sesion.access_token)
        resultado = await services.equipment.crear(
            tipo='Laptop', marca='Marca', modelo='Modelo',
            problema_reportado='No enciende', client_id=7, numero_serie='DEMO-001')
        equipos = await services.equipment.listar()
        assert equipos[0].numero_serie == 'DEMO-001'
        print('Folio:', resultado.folio)
        print('Equipos consultados:', len(equipos))
        print('Solicitudes (método, ruta, tiene autorización):', solicitudes)
        print('Transporte compartido:', services.clients.api is services.equipment.api)
    finally:
        await api.aclose()
    assert api._client is None
    print('HTTP cerrado.')

await ejemplo_alta_y_listado()

Folio: EQ-001
Equipos consultados: 1
Solicitudes (m?todo, ruta, tiene autorizaci?n): [('POST', '/auth/login', False), ('POST', '/equipos/', True), ('GET', '/equipos/', True)]
Transporte compartido: True
HTTP cerrado.


In [3]:
async def ejemplo_errores():
    def servidor(request):
        return httpx.Response(422, json={'detail': [{'msg': 'Nombre obligatorio'}]})

    api = ApiClient(transport=httpx.MockTransport(servidor))
    try:
        await api.request('POST', '/clientes/', json={})
    except ApiError as error:
        print('Mensaje visible:', str(error))
        print('Código HTTP:', error.status_code)
        assert error.status_code == 422
    finally:
        await api.aclose()

await ejemplo_errores()

Mensaje visible: Nombre obligatorio
C?digo HTTP: 422


In [4]:
async def ejemplo_concurrencia():
    iniciadas = set()
    todas_iniciadas = asyncio.Event()

    async def servidor(request):
        iniciadas.add(request.url.path)
        if len(iniciadas) == 2:
            todas_iniciadas.set()
        # Ambos GET deben comenzar antes de que cualquiera finalice.
        await asyncio.wait_for(todas_iniciadas.wait(), timeout=1)
        return httpx.Response(200, json=[])

    api = ApiClient(transport=httpx.MockTransport(servidor))
    services = Services(api)
    try:
        clientes, equipos = await asyncio.gather(
            services.clients.listar(), services.equipment.listar())
        assert len(iniciadas) == 2
        print('Consultas concurrentes:', sorted(iniciadas))
        print('Resultados:', clientes, equipos)
    finally:
        await api.aclose()

await ejemplo_concurrencia()

Consultas concurrentes: ['/clientes/', '/equipos/']
Resultados: [] []


## Contrato de modelos → actualización del 02-Oct-2026

Los services ya no retornan diccionarios de dominio. Retornan modelos Pydantic con atributos validados. El transporte ApiClient sigue retornando JSON decodificado; ModuleService._parse aplica el contrato y transforma ValidationError en ApiError sin mostrar los datos inválidos.

```text
Formulario → modelo Crear/Login → model_dump() → HTTP
HTTP → JSON → modelo de respuesta → atributos en la vista
```

| Módulo | Entradas | Consultas | Confirmaciones |
|---|---|---|---|
| auth | Login | Sesion | Sesion con token_type bearer |
| clients | ClienteCrear | Cliente | ClienteRegistrado: mensaje + cliente |
| equipment | EquipoCrear | Equipo | EquipoRegistrado: mensaje + folio + DetallesEquipo |
| users | UsuarioCrear, RolActualizar | Usuario | UsuarioRegistrado o Mensaje |

Ejemplo de uso:

```python
equipos = await services.equipment.listar()
print(equipos[0].numero_serie)
sesion = await services.auth.login(username, password)
api.set_token(sesion.access_token)
```

`Contrato` hereda de BaseModel y configura strict=True, frozen=True y extra='ignore': valida tipos sin convertir cadenas a enteros/booleanos, evita modificaciones accidentales y descarta campos no declarados. Un ID debe ser entero positivo: enviar 7, no '7'. Los opcionales aceptan null/None o ausencia y tienen default None. Los roles están restringidos a admin, tecnico y recepcion con Literal.

Los modelos de alta incluyen las longitudes existentes del backend; correo se declara como string opcional y su formato lo valida el servidor. La autenticación, unicidad e integridad siguen correspondiendo al backend. Ignorar campos adicionales permite evolucionar respuestas sin conservar hashes de usuarios; no soluciona que un backend los exponga en la red.

Pydantic se usa de forma declarativa: campos, tipos, Field y dos operaciones (`model_validate` / `model_dump`). No hay validadores personalizados, clases ORM ni conexión de BD en frontend. La representación de password/access_token se oculta con repr=False; model_dump aún contiene sus valores para enviar credenciales, por lo que no debe imprimirse en logs reales.

Las confirmaciones de alta también se validan: un HTTP 201 con JSON incompleto genera error de contrato y no se muestra como éxito confirmado. Puede haber una escritura ya guardada en servidor: consultar antes de repetir un alta.

Referencias: [modelos Pydantic](https://docs.pydantic.dev/latest/concepts/models/) y [modo estricto](https://docs.pydantic.dev/latest/concepts/strict_mode/).


## Tutorial: integrar un nuevo módulo

Versión con runner encapsulado. Disponible como [notebook independiente](GUIA_INTEGRACION_MODULOS_FRONTEND.ipynb).

# Integrar módulos con el runner encapsulado

## 1. Crear únicamente tres archivos

| Archivo nuevo | Responsabilidad |
|---|---|
| `frontend/src/models/orders.py` | Contratos de entrada y salida. |
| `frontend/src/services/orders_service.py` | Peticiones y validación del dominio. |
| `frontend/src/ui/views/orders.py` | Campos, presentación y permisos del menú. |

No editar Services, SistemaGestion, login ni AsyncRunner. Los servicios se descubren por el sufijo `_service.py`, y las vistas por sus metadatos. Reiniciar la app después de añadir archivos. Los módulos se importan al arrancar: no colocar scripts con efectos secundarios en estas carpetas.

Arrancar desde la raíz con `python frontend/src/app/main.py`. El punto de entrada incorpora `frontend/src` al recorrido de imports, por eso se usan `models`, `services` y `ui` sin el prefijo `frontend.src`.

El ejemplo propone GET `/ordenes/` y POST `/ordenes/`. Estos endpoints aún deben implementarse en el backend. El servidor valida equipo, asigna estado inicial y autoriza cada operación; revisar requisitos, SQL y diagrama antes de adoptar este contrato.

## 2. Modelo: contenido de models/orders.py

```python
from pydantic import Field
from models.base import Contrato, Mensaje


class OrdenCrear(Contrato):
    """equipo_id: identificador de un equipo existente."""
    equipo_id: int = Field(gt=0)


class Orden(Contrato):
    """Registro público del servidor."""
    id: int = Field(gt=0)
    equipo_id: int = Field(gt=0)
    estado: str


class OrdenRegistrada(Mensaje):
    """Confirmación: mensaje heredado y orden creada."""
    orden: Orden
```

`Field` aquí es Pydantic; `gt=0` exige un entero positivo. `Contrato` valida tipos estrictos sin convertir texto a entero, ignora extras y hace inmutables los datos. `Mensaje` exige `mensaje` no vacío. Separar entrada y salida permite que el servidor asigne ID y estado.

## 3. Servicio: contenido de services/orders_service.py

```python
from services.api_client import ModuleService
from models.orders import OrdenCrear, Orden, OrdenRegistrada


class OrdersService(ModuleService):
    """Operaciones de órdenes con el transporte compartido."""
    service_name = 'orders'

    async def listar(self) -> list[Orden]:
        return await self._list('/ordenes/', Orden)

    async def crear(self, equipo_id) -> OrdenRegistrada:
        entrada = self._parse(OrdenCrear, {'equipo_id': equipo_id}, entrada=True)
        data = await self.api.request(
            'POST', '/ordenes/', json=entrada.model_dump(mode='json'))
        return self._parse(OrdenRegistrada, data)
```

| Elemento | Parámetros y motivo |
|---|---|
| `service_name` | Clave pública `services.orders`; si se omite, se usa el nombre del archivo sin `_service.py`. |
| Constructor heredado `__init__(api)` | Recibe el ApiClient existente; no crear HTTP ni runner nuevos. |
| `listar()` | Retorna lista de modelos, requerida por la consulta común de la vista. |
| `crear(equipo_id)` | Recibe entero positivo ya convertido por la vista. |
| `_list(path, model)` | Ruta relativa de GET y contrato de cada registro; exige lista JSON. |
| `_parse(model, data, entrada=True)` | Contrato y datos; identifica errores de formulario. |
| `_parse(OrdenRegistrada, data)` | Valida confirmación antes de mostrar éxito. |
| `request(method, path, json=...)` | Método, ruta y cuerpo; ApiClient aplica JWT y traduce errores HTTP. |

El archivo debe terminar en `_service.py` y definir una clase que herede ModuleService. Services importa esa clase y la construye con la API compartida; no registra otra vez clases importadas desde otro archivo. La clave debe ser única, identificador Python válido y no reservada. Claves duplicadas provocan error al iniciar.

Usar `params` para filtros y `json` para cuerpos JSON. `mode='json'` serializa tipos como fechas cuando se añadan. Dejar que ApiError llegue al callback visual: devolver una lista vacía ante un error ocultaría el fallo como ausencia de datos.

## 4. Vista: contenido de ui/views/orders.py

```python
from models.orders import Orden
from services.api_client import ApiError
from ui.components import CollectionView, Field


class OrdersView(CollectionView):
    """Listado y alta; la base administra solicitudes y formularios."""
    menu_label = 'Órdenes'
    allowed_roles = ('admin', 'recepcion')
    menu_order = 25
    service_name = 'orders'
    title = 'Órdenes de servicio'
    columns = ('ID', 'ID Equipo', 'Estado')
    fields = (Field('equipo_id', 'ID del equipo'),)

    async def save(self, values):
        try:
            equipo_id = int(values['equipo_id'])
        except ValueError:
            raise ApiError('El ID del equipo debe ser un entero.') from None
        return await self.service.crear(equipo_id=equipo_id)

    def row(self, record: Orden):
        return (record.id, record.equipo_id, record.estado)
```

| Miembro | Propósito |
|---|---|
| `menu_label` | Texto del botón; sin etiqueta no se incorpora al menú. |
| `allowed_roles` | Roles API que pueden ver la pantalla; el backend también verifica permisos. |
| `menu_order` | Orden ascendente; empate por etiqueta; valor predeterminado 100. |
| `service_name` | Conecta la vista con `services.orders`. |
| `title` | Encabezado y título del formulario. |
| `columns` | Encabezados, en el orden de `row`. |
| `fields` | Claves y etiquetas del formulario. Aquí Field es de UI, no Pydantic. |
| `save(values)` | Diccionario capturado por el formulario; convierte ID explícitamente. |
| `row(record)` | Modelo leído; devuelve valores de presentación, sin consultar HTTP. |

No hace falta escribir `fetch`: la base llama a `self.service.listar()`. Su `save` predeterminado llama a `self.service.crear(**values)`; aquí lo sobrescribimos porque Entry entrega texto y el contrato exige entero. Sobrescribir fetch solo si se necesitan filtros o consultas propias.

El constructor heredado recibe `master` (contenedor Tk), `services` (servicios compartidos), `runner` (infraestructura interna) y `role` (rol de sesión). No definirlo si no se agregan controles; si se redefine, conservar la firma y llamar a super. La base carga al construirse: preparar previamente atributos utilizados por fetch.

`Field(name, label, required=True, secret=False, choices=())`: name identifica la clave de values, label el texto, required exige presencia, secret oculta y conserva espacios, choices genera selector. `EquipmentView` muestra un selector que traduce etiquetas a IDs.

La base administra carga, formulario, errores, Actualizar y recarga después del alta. La confirmación debe ser compatible con Mensaje. El folio mostrado por la base sigue siendo específico de equipos. No duplicar submit ni crear tareas en fetch/save.

## 5. Operación personalizada sin conocer el runner

Para una pantalla distinta de listado/alta, heredar AsyncFrame y construir controles. Usar este patrón dentro de un método normal de la vista:

```python
try:
    equipo_id = int(self.entry_equipo.get())
except ValueError:
    self.status.configure(text='El ID debe ser un entero.')
    return
self.button.configure(state='disabled')
self.request(self.services.orders.crear, equipo_id,
             on_success=self._saved, on_error=self._failed)
```

`request(operation, *args, on_success, on_error=None, **kwargs)` recibe el método async sin llamarlo, los datos y callbacks normales. La base crea la coroutine, la vincula a esta vista y programa su ejecución. `on_success(result)` recibe el resultado; `on_error(error)` recibe una excepción. Si se omite on_error, se usa `_failed`, que muestra el error en `self.status`. Si se deshabilitan botones, definir `_saved` y `_failed` para restaurarlos.

Leer widgets antes de request. Los services no acceden a Tk; los callbacks sí corren en su hilo principal. AsyncFrame cancela tareas al destruirse; un destroy propio debe llamar a super. El Future retornado solo es necesario para cancelar una lectura específica; no esperar con `.result()` en Tk ni usar asyncio.run en botones.

El hilo y la cola permanecen encapsulados. Cambiar de vista cancela sus tareas, no cierra la API compartida. Cerrar la aplicación sigue siendo responsabilidad del login. Cancelar una petición no revierte un POST ya recibido. La revisión de la cola con after no consulta Supabase; Actualizar es una nueva lectura, sin suscripción en tiempo real.

## 6. Validación y errores frecuentes

1. Reiniciar la app y comprobar el menú con cada rol.
2. Comprobar que Services expone orders y comparte api con otros servicios.
3. Usar MockTransport siguiendo el laboratorio v2: GET y POST válidos, payload, entrada inválida sin HTTP, error 403 y respuesta malformada.
4. Comprobar row con una Orden y abrir/cerrar mientras una consulta está pendiente.
5. Ejecutar `python -m unittest discover -s frontend/tests -v` y después probar contra el backend real.

Los tests de `test_module_integration.py` verifican descubrimiento sin editar contenedores, roles, cliente compartido, claves duplicadas y delegación de operaciones. No certifican los endpoints de órdenes ni la apariencia visual.

| Síntoma | Revisar |
|---|---|
| Servicio ausente | Sufijo `_service.py`, herencia y service_name. |
| Vista ausente | Archivo en ui/views, herencia AsyncFrame, etiqueta y roles. |
| Error al arrancar | Importación incorrecta o clave duplicada/reservada. |
| Error de contrato | Tipos estrictos o JSON incompatible. |
| HTTP 404 | Router de órdenes pendiente o ruta distinta. |
| HTTP 403 | Permisos del backend. |
| Interfaz congelada | Trabajo bloqueante en Tk; evitar `.result()` y asyncio.run. |

## 15. Inventario y código vigente

Instantáneas actuales, con descubrimiento y runner encapsulado.

### `frontend/src/app/__init__.py`

Marcador de paquete.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.



### `frontend/src/app/async_runner.py`

Puente asyncio/Tkinter: red en un hilo; todos los widgets en el hilo Tk.

SHA-256: `632e297393a2ebca01acbda5534ab434d879f38490d1c75de10b65b1f560c79c`.

- L16: `__init__(self, root)`: root: ventana principal que programa la revisión de resultados.
- L34: `_run(self)`: Mantiene un único loop para todas las conexiones HTTP.
- L45: `submit(self, coroutine, owner, on_success, on_error)`: Programar desde Tk y devolver un Future cancelable (None al cerrar).

coroutine: llamada async ya creada, por ejemplo service.listar().
owner: widget cuya destrucción debe cancelar la entrega del resultado.
on_success(result) y on_error(exception): funciones normales en Tk.
No leer widgets dentro de coroutine: capturar sus valores antes de submit.
- L64: `cancel_owner(self, owner)`: owner: vista/modal cerrado; elimina sus callbacks y cancela peticiones.
- L74: `_poll(self)`: Procesa resultados en el hilo principal; ignora ventanas destruidas.
- L104: `close(self, cleanup, on_closed)`: cleanup: coroutine que cierra HTTP; on_closed: cierre de ventana en Tk.
- L152: `_wait_thread(self, on_closed)`: on_closed: se llama cuando el trabajador ya terminó.
- L121: `shutdown()`: Espera cancelaciones y libera HTTP antes de detener el loop.
- L135: `finish()`: Revisa el cierre sin bloquear el bucle de eventos de Tk.

**Código de referencia**

```python
"""Puente asyncio/Tkinter: red en un hilo; todos los widgets en el hilo Tk."""
import asyncio
from concurrent.futures import CancelledError
from queue import Empty, Queue
from threading import Event, Thread
import logging


class AsyncRunner:
    """Puente compartido: recibe coroutines y entrega sus resultados al hilo Tk.

    Crear una sola instancia desde login y pasarla a las vistas. Las coroutines
    hacen I/O; los callbacks modifican widgets. No usar asyncio.run en botones.
    """

    def __init__(self, root):
        """root: ventana principal que programa la revisión de resultados."""

        self.root = root
        self.loop = asyncio.new_event_loop()
        self._ready = Event()
        # El trabajador produce resultados; Tk consume la cola sin bloquear.
        self._queue = Queue()
        # Future -> (propietario, callback de éxito, callback de error).
        # Solo el hilo Tk modifica este registro.
        self._pending = {}
        self._closing = False
        self._thread = Thread(target=self._run, name='fixify-asyncio', daemon=True)
        self._thread.start()
        self._ready.wait()

        self._poll_id = root.after(30, self._poll)

    def _run(self):
        """Mantiene un único loop para todas las conexiones HTTP."""
        asyncio.set_event_loop(self.loop)
        self._ready.set()

        try:
            self.loop.run_forever()
        finally:
            self.loop.run_until_complete(self.loop.shutdown_asyncgens())
            self.loop.close()

    def submit(self, coroutine, owner, on_success, on_error):
        """Programar desde Tk y devolver un Future cancelable (None al cerrar).

        coroutine: llamada async ya creada, por ejemplo service.listar().
        owner: widget cuya destrucción debe cancelar la entrega del resultado.
        on_success(result) y on_error(exception): funciones normales en Tk.
        No leer widgets dentro de coroutine: capturar sus valores antes de submit.
        """

        if self._closing:
            coroutine.close()
            return None

        future = asyncio.run_coroutine_threadsafe(coroutine, self.loop)
        self._pending[future] = (owner, on_success, on_error)
        # Este callback solo toca una cola Python; nunca llama a Tkinter.
        future.add_done_callback(self._queue.put)
        return future

    def cancel_owner(self, owner):
        """owner: vista/modal cerrado; elimina sus callbacks y cancela peticiones."""

        # Cancelar una petición no garantiza deshacer una escritura ya recibida
        # por el servidor. No reintentar POST automáticamente tras cancelación.
        for future, context in list(self._pending.items()):
            if context[0] is owner:
                self._pending.pop(future, None)
                future.cancel()

    def _poll(self):
        """Procesa resultados en el hilo principal; ignora ventanas destruidas."""

        try:
            while True:
                future = self._queue.get_nowait()
                context = self._pending.pop(future, None)

                if context is None:
                    continue

                owner, success, error = context

                if not owner.winfo_exists():
                    continue

                try:
                    result = future.result()
                except CancelledError:
                    continue
                except Exception as exc:
                    error(exc)
                else:
                    success(result)
        except Empty:
            pass
        finally:
            if not self._closing:
                self._poll_id = self.root.after(30, self._poll)

    def close(self, cleanup, on_closed):
        """cleanup: coroutine que cierra HTTP; on_closed: cierre de ventana en Tk."""

        # Solo el cierre global llama a este método. Cambiar de vista utiliza
        # cancel_owner; cerrar el cliente aquí lo inutilizaría para otros módulos.
        if self._closing:
            cleanup.close()
            return

        self._closing = True
        self.root.after_cancel(self._poll_id)

        for future in self._pending:
            future.cancel()

        self._pending.clear()

        async def shutdown():
            """Espera cancelaciones y libera HTTP antes de detener el loop."""

            current = asyncio.current_task()
            tasks = [task for task in asyncio.all_tasks() if task is not current]

            for task in tasks:
                task.cancel()

            await asyncio.gather(*tasks, return_exceptions=True)
            await cleanup

        future = asyncio.run_coroutine_threadsafe(shutdown(), self.loop)

        def finish():
            """Revisa el cierre sin bloquear el bucle de eventos de Tk."""

            if not future.done():
                self.root.after(30, finish)
                return

            try:
                future.result()
            except Exception:
                logging.exception('No se pudo cerrar el cliente HTTP')

            self.loop.call_soon_threadsafe(self.loop.stop)
            self._wait_thread(on_closed)

        self.root.after(30, finish)

    def _wait_thread(self, on_closed):
        """on_closed: se llama cuando el trabajador ya terminó."""

        if self._thread.is_alive():
            self.root.after(30, lambda: self._wait_thread(on_closed))
        else:
            on_closed()

```

### `frontend/src/app/main.py`

Entrada de escritorio: python frontend/src/app/main.py desde la raíz.

SHA-256: `edc8a4bd6c81f70f4aacf64c2e15ec0b994d7a3ac3b1a448b0bf5ba9ee109f16`.



**Código de referencia**

```python
"""Entrada de escritorio: python frontend/src/app/main.py desde la raíz."""
from pathlib import Path
import sys

# Permite imports de app, services y ui al ejecutar este archivo directamente.
RUTA_SRC = str(Path(__file__).resolve().parents[1])
if RUTA_SRC not in sys.path:
    sys.path.insert(0, RUTA_SRC)

from ui.login import VentanaLogin


if __name__ == '__main__':
    app = VentanaLogin()
    app.mainloop()

```

### `frontend/src/models/__init__.py`

Contratos Pydantic de entrada/salida, separados por módulo de negocio.

SHA-256: `0f07cbabe644a614588ea50219eb23d87f4dd2c7b75ad4d4036861bdba357eeb`.



**Código de referencia**

```python
"""Contratos Pydantic de entrada/salida, separados por módulo de negocio."""

```

### `frontend/src/models/auth.py`

Credenciales de entrada y sesión devuelta por /auth/login.

SHA-256: `5b0bb869e5fc39092b280ab9919e289d4eb3e732fbd8eab40fe516e1c5094e51`.



**Código de referencia**

```python
"""Credenciales de entrada y sesión devuelta por /auth/login."""
from typing import Literal
from pydantic import Field
from models.base import Contrato, Rol


class Login(Contrato):
    """username/password: credenciales; password se oculta en la representación."""
    username: str = Field(min_length=1)
    password: str = Field(min_length=1, repr=False)


class Sesion(Contrato):
    """access_token: JWT; token_type: esquema Bearer; role: permisos; message: confirmación."""
    access_token: str = Field(min_length=1, repr=False)
    token_type: Literal['bearer']
    role: Rol
    message: str | None = None

```

### `frontend/src/models/base.py`

Configuración común del contrato; no contiene acceso HTTP ni a la BD.

SHA-256: `062551390290430148348056b390c9d7e6db48f804048b267c7b2e4d61cb8483`.



**Código de referencia**

```python
"""Configuración común del contrato; no contiene acceso HTTP ni a la BD."""
from typing import Literal
from pydantic import BaseModel, ConfigDict, Field

Rol = Literal['admin', 'tecnico', 'recepcion']


class Contrato(BaseModel):
    """Valida tipos sin conversiones silenciosas e ignora campos extra del servidor."""
    model_config = ConfigDict(strict=True, extra='ignore', frozen=True)


class Mensaje(Contrato):
    """Confirmación de operaciones: mensaje es el texto enviado por el backend."""
    mensaje: str = Field(min_length=1)

```

### `frontend/src/models/clients.py`

Contrato de clientes: datos para alta, registro consultado y confirmación.

SHA-256: `45da089d667c597b4dbe6e973c0ef4ccd7e6c2325cdd6aa42624a9749f5ffb82`.



**Código de referencia**

```python
"""Contrato de clientes: datos para alta, registro consultado y confirmación."""
from pydantic import Field
from models.base import Contrato, Mensaje


class ClienteCrear(Contrato):
    """nombre/telefono: obligatorios; correo/direccion: contacto opcional."""
    nombre: str = Field(min_length=2, max_length=100)
    telefono: str = Field(min_length=8, max_length=20)
    correo: str | None = None
    direccion: str | None = Field(default=None, max_length=250)


class Cliente(Contrato):
    """Registro leído; id es el identificador API, no el nombre físico de columna."""
    id: int = Field(gt=0)
    nombre: str
    telefono: str
    correo: str | None = None
    direccion: str | None = None


class ClienteRegistrado(Mensaje):
    """Confirmación de alta; cliente contiene el registro persistido."""
    cliente: Cliente

```

### `frontend/src/models/equipment.py`

Contrato de equipos; mantiene los nombres públicos de la API.

SHA-256: `02a5e221d23a0ee093903144d060f8e03bc4f9af4d473a7b7d27babab987aca7`.



**Código de referencia**

```python
"""Contrato de equipos; mantiene los nombres públicos de la API."""
from pydantic import Field
from models.base import Contrato, Mensaje


class EquipoCrear(Contrato):
    """Características, falla y client_id del propietario; serie opcional."""
    tipo: str = Field(min_length=2, max_length=50)
    marca: str = Field(min_length=2, max_length=50)
    modelo: str = Field(min_length=2, max_length=50)
    problema_reportado: str = Field(min_length=5)
    client_id: int = Field(gt=0)
    numero_serie: str | None = Field(default=None, max_length=100)


class Equipo(Contrato):
    """Registro consultado: id, características, falla y propietario."""
    id: int = Field(gt=0)
    tipo: str
    marca: str
    modelo: str
    problema_reportado: str
    client_id: int = Field(gt=0)
    numero_serie: str | None = None


class DetallesEquipo(Contrato):
    """Resumen del POST; usa id_interno/cliente_id según la respuesta actual."""
    id_interno: int = Field(gt=0)
    tipo: str
    marca: str
    modelo: str
    cliente_id: int = Field(gt=0)


class EquipoRegistrado(Mensaje):
    """Confirmación de alta: folio visual y resumen del equipo guardado."""
    folio: str = Field(min_length=1)
    detalles: DetallesEquipo

```

### `frontend/src/models/users.py`

Contrato de personal, alta y cambios de rol; nunca conserva hashes.

SHA-256: `22592fb3e3a0e0eaf70c1f17f70e0b698afb6fafbec7e289c11f84b7637ded2e`.



**Código de referencia**

```python
"""Contrato de personal, alta y cambios de rol; nunca conserva hashes."""
from pydantic import Field
from models.base import Contrato, Mensaje, Rol


class UsuarioCrear(Contrato):
    """username/password: credenciales; role: rol autorizado de dominio."""
    username: str = Field(min_length=4, max_length=50)
    password: str = Field(min_length=6, repr=False)
    role: Rol = 'tecnico'


class Usuario(Contrato):
    """Registro público: ID, nombre de cuenta, rol y estado de acceso."""
    id: int = Field(gt=0)
    username: str
    role: Rol
    is_active: bool


class RolActualizar(Contrato):
    """role: nuevo rol de la cuenta seleccionada."""
    role: Rol


class UsuarioRegistrado(Mensaje):
    """usuario: username confirmado por POST /usuarios/."""
    usuario: str

```

### `frontend/src/services/__init__.py`

Construye los servicios con una sola sesión HTTP compartida.

SHA-256: `4520ede20b00670dd5fd0eb4ec441b6f08d156d7134f9ae3c402360197e1ca6d`.

- L9: `__init__(self, api=None)`: api: transporte opcional; si se omite utiliza la configuración local.

**Código de referencia**

```python
"""Construye los servicios con una sola sesión HTTP compartida."""
from services.api_client import ApiClient
from services.discovery import service_classes


class Services:
    """Contenedor inyectable para evitar clientes globales en las vistas."""

    def __init__(self, api=None):
        """api: transporte opcional; si se omite utiliza la configuración local."""

        self.api = api if api is not None else ApiClient()

        # Cada archivo *_service.py aporta un ModuleService. Todos reciben la
        # misma API; añadir un servicio no exige editar este contenedor.
        for name, service_class in service_classes().items():
            if hasattr(self, name):
                raise ValueError(f'Nombre de servicio reservado: {name}')

            setattr(self, name, service_class(self.api))

```

### `frontend/src/services/api_client.py`

Transporte HTTP compartido; las rutas de negocio viven en cada service.

SHA-256: `7494c1facdb48289722a545a9b86bd5e9bacbbb053351228c5fa631744a17112`.

- L16: `__init__(self, message, status_code=None)`: Sin docstring.
- L25: `__init__(self, base_url=None, transport=None)`: base_url: servidor; transport: transporte opcional para pruebas sin red.
- L33: `set_token(self, token)`: token: JWT vigente o None para cerrar la sesión local.
- L38: `request(self, method, path, *, authenticated=True, **kwargs)`: method/path: operación HTTP (PUT, POST, GET, DELETE); kwargs: JSON, formulario o parámetros.
- L80: `_error_message(response)`: response: respuesta HTTP; convierte detail de FastAPI a texto breve.
- L96: `aclose(self)`: Libera conexiones al cerrar la aplicación.
- L113: `__init__(self, api)`: api: ApiClient compartido por los módulos.
- L119: `_parse(model: type[Modelo], data, *, entrada=False)`: model: contrato esperado; data: valores; entrada: distingue formulario de respuesta.
- L129: `_list(self, path, model: type[Modelo], **kwargs)`: path: colección; model: contrato de cada registro; kwargs: filtros opcionales.

**Código de referencia**

```python
"""Transporte HTTP compartido; las rutas de negocio viven en cada service."""
import os
from typing import TypeVar

import httpx
from pydantic import ValidationError

from models.base import Contrato

Modelo = TypeVar('Modelo', bound=Contrato)


class ApiError(Exception):
    """Error legible de API; status_code identifica errores HTTP, si existen."""

    def __init__(self, message, status_code=None):
        super().__init__(message)

        self.status_code = status_code


class ApiClient:
    """Reutiliza conexiones HTTP y el token de la sesión de escritorio."""

    def __init__(self, base_url=None, transport=None):
        """base_url: servidor; transport: transporte opcional para pruebas sin red."""

        self.base_url = base_url or os.getenv('FIXIFY_API_URL', 'http://127.0.0.1:8000')
        self.token = None
        self._transport = transport
        self._client = None

    def set_token(self, token):
        """token: JWT vigente o None para cerrar la sesión local."""

        self.token = token

    async def request(self, method, path, *, authenticated=True, **kwargs):
        """method/path: operación HTTP (PUT, POST, GET, DELETE); kwargs: JSON, formulario o parámetros."""

        # Se crea y utiliza exclusivamente en el loop asyncio del trabajador.
        if self._client is None:
            self._client = httpx.AsyncClient(
                base_url=self.base_url,
                timeout=5,
                transport=self._transport,
                follow_redirects=True,
            )

        headers = (
            {'Authorization': f'Bearer {self.token}'}
            if authenticated and self.token
            else {}
        )

        try:
            response = await self._client.request(
                method,
                path,
                headers=headers,
                **kwargs,
            )
        except httpx.TimeoutException as exc:
            raise ApiError('El servidor tardó demasiado en responder.') from exc
        except httpx.RequestError as exc:
            raise ApiError('No se pudo conectar con el servidor.') from exc

        if response.is_error:
            raise ApiError(self._error_message(response), response.status_code)

        if response.status_code == 204:
            return None

        try:
            return response.json()
        except ValueError as exc:
            raise ApiError('El servidor devolvió una respuesta inválida.') from exc

    @staticmethod
    def _error_message(response):
        """response: respuesta HTTP; convierte detail de FastAPI a texto breve."""

        try:
            detail = response.json().get('detail')

            if isinstance(detail, list):
                return '\n'.join(str(error.get('msg', 'Dato inválido')) for error in detail)

            if isinstance(detail, str):
                return detail
        except (ValueError, AttributeError, TypeError):
            pass

        return f'La solicitud falló (HTTP {response.status_code}).'

    async def aclose(self):
        """Libera conexiones al cerrar la aplicación."""

        if self._client is not None:
            await self._client.aclose()

            self._client = None


class ModuleService:
    """Base de servicios de dominio; no accede a widgets ni depende de Tk.

    Para otro módulo: heredar, usar _list(ruta, Modelo) en GET y validar
    entradas con _parse(..., entrada=True) antes de enviar JSON. Validar
    también la respuesta con _parse; dejar que ApiError llegue a la vista.
    """

    def __init__(self, api):
        """api: ApiClient compartido por los módulos."""

        self.api = api

    @staticmethod
    def _parse(model: type[Modelo], data, *, entrada=False) -> Modelo:
        """model: contrato esperado; data: valores; entrada: distingue formulario de respuesta."""

        try:
            return model.model_validate(data)
        except ValidationError:
            # No incluimos valores inválidos: podrían contener contraseñas o tokens.
            origen = 'Los datos ingresados' if entrada else 'La respuesta del servidor'
            raise ApiError(f'{origen} no cumplen el contrato {model.__name__}.') from None

    async def _list(self, path, model: type[Modelo], **kwargs) -> list[Modelo]:
        """path: colección; model: contrato de cada registro; kwargs: filtros opcionales."""

        data = await self.api.request('GET', path, **kwargs)

        if not isinstance(data, list) or any(
            not isinstance(item, dict) for item in data
        ):
            raise ApiError('El servidor no devolvió una lista válida.')

        return [self._parse(model, item) for item in data]

```

### `frontend/src/services/auth_service.py`

Operaciones de autenticación; no modifica widgets ni guarda contraseñas.

SHA-256: `4dc8a4bad62ccc3e8fb5621eb9915d34d209ea4ee2792a708c367c1c66c6ddc9`.

- L9: `login(self, username, password)`: username/password: credenciales; retorna token y rol validados.

**Código de referencia**

```python
"""Operaciones de autenticación; no modifica widgets ni guarda contraseñas."""
from services.api_client import ModuleService
from models.auth import Login, Sesion


class AuthService(ModuleService):
    """Solicita credenciales de sesión al backend."""

    async def login(self, username, password) -> Sesion:
        """username/password: credenciales; retorna token y rol validados."""

        entrada = self._parse(
            Login,
            {'username': username, 'password': password},
            entrada=True,
        )
        data = await self.api.request(
            'POST',
            '/auth/login',
            authenticated=False,
            data=entrada.model_dump(),
        )

        # La UI guarda el token solo si aún existe la ventana que pidió el login.
        return self._parse(Sesion, data)

```

### `frontend/src/services/clients_service.py`

Rutas de clientes y transformación de sus datos de entrada.

SHA-256: `59461a3322e12c6cc3b48f518919b7f41d262dac7b63331e751d72245ed76a06`.

- L9: `listar(self)`: Retorna los clientes permitidos para la sesión actual.
- L13: `crear(self, nombre, telefono, correo=None, direccion=None)`: nombre/telefono: obligatorios; correo/direccion: datos opcionales.
- L24: `buscar(self, termino)`: termino: parte del nombre o teléfono que se desea localizar.

**Código de referencia**

```python
"""Rutas de clientes y transformación de sus datos de entrada."""
from services.api_client import ModuleService
from models.clients import Cliente, ClienteCrear, ClienteRegistrado


class ClientsService(ModuleService):
    """Consulta y registra clientes mediante la API."""

    async def listar(self) -> list[Cliente]:
        """Retorna los clientes permitidos para la sesión actual."""
        return await self._list('/clientes/', Cliente)

    async def crear(self, nombre, telefono, correo=None, direccion=None) -> ClienteRegistrado:
        """nombre/telefono: obligatorios; correo/direccion: datos opcionales."""

        entrada = self._parse(ClienteCrear, {
            'nombre': nombre, 'telefono': telefono,
            'correo': correo or None, 'direccion': direccion or None,
        }, entrada=True)
        data = await self.api.request('POST', '/clientes/', json=entrada.model_dump())

        return self._parse(ClienteRegistrado, data)

    async def buscar(self, termino) -> list[Cliente]:
        """termino: parte del nombre o teléfono que se desea localizar."""
        return await self._list('/clientes/buscar', Cliente, params={'q': termino})

```

### `frontend/src/services/discovery.py`

Encuentra servicios locales para construir el contenedor Services.

Cada archivo *_service.py define una clase de ModuleService. service_name
indica su atributo público; si falta, se usa el nombre del archivo sin sufijo.
Solo descubre clases: Services las instancia con el ApiClient compartido.
Los imports ejecutan código de módulo; mantener estos archivos sin efectos
secundarios. Un error de importación se propaga para detectar fallos al iniciar.

SHA-256: `c36482a44b492e8835ec7976dd2279c78924a530cfc4b24b9a6bd399d4bb14f3`.

- L15: `service_classes()`: Retorna dict de clave pública -> clase; no recibe parámetros.

Lanza ValueError para claves inválidas o duplicadas. Una clase importada
desde otro módulo se ignora para no registrar el mismo servicio dos veces.
Ejemplo: orders_service.py + OrdersService -> {'orders': OrdersService}.

**Código de referencia**

```python
"""Encuentra servicios locales para construir el contenedor Services.

Cada archivo *_service.py define una clase de ModuleService. service_name
indica su atributo público; si falta, se usa el nombre del archivo sin sufijo.
Solo descubre clases: Services las instancia con el ApiClient compartido.
Los imports ejecutan código de módulo; mantener estos archivos sin efectos
secundarios. Un error de importación se propaga para detectar fallos al iniciar.
"""
from importlib import import_module
from pkgutil import iter_modules

from services.api_client import ModuleService


def service_classes():
    """Retorna dict de clave pública -> clase; no recibe parámetros.

    Lanza ValueError para claves inválidas o duplicadas. Una clase importada
    desde otro módulo se ignora para no registrar el mismo servicio dos veces.
    Ejemplo: orders_service.py + OrdersService -> {'orders': OrdersService}.
    """
    # Import local: Services importa discovery durante su propia inicialización.
    import services

    found = {}

    for entry in sorted(iter_modules(services.__path__), key=lambda item: item.name):
        if not entry.name.endswith('_service'):
            continue

        module = import_module(f'services.{entry.name}')

        # __module__ distingue las clases propias de las que solo se importaron.
        for value in vars(module).values():
            if (
                isinstance(value, type)
                and issubclass(value, ModuleService)
                and value.__module__ == module.__name__
            ):
                name = getattr(
                    value,
                    'service_name',
                    entry.name.removesuffix('_service'),
                )

                # La clave debe poder usarse como services.nombre y no ocultar api.
                if (
                    not isinstance(name, str)
                    or not name.isidentifier()
                    or name.startswith('_')
                    or name == 'api'
                ):
                    raise ValueError(f'Nombre de servicio inválido: {name!r}')

                if name in found:
                    raise ValueError(f'Servicio duplicado: {name}')

                found[name] = value

    return found

```

### `frontend/src/services/equipment_service.py`

Operaciones del módulo de equipos; mantiene el contrato actual de API.

SHA-256: `d9c6b58e369208a3cb404c1633856e5c0a82cd232272b18a5120a381d513f473`.

- L9: `listar(self)`: Retorna equipos mediante GET /equipos/.
- L13: `crear(self, tipo, marca, modelo, problema_reportado, client_id, numero_serie=None)`: tipo/marca/modelo: equipo; problema_reportado: falla; client_id: dueño; numero_serie: opcional.

**Código de referencia**

```python
"""Operaciones del módulo de equipos; mantiene el contrato actual de API."""
from services.api_client import ModuleService
from models.equipment import Equipo, EquipoCrear, EquipoRegistrado


class EquipmentService(ModuleService):
    """Consulta y registra equipos asociados a un cliente."""

    async def listar(self) -> list[Equipo]:
        """Retorna equipos mediante GET /equipos/."""
        return await self._list('/equipos/', Equipo)

    async def crear(
        self,
        tipo,
        marca,
        modelo,
        problema_reportado,
        client_id,
        numero_serie=None,
    ) -> EquipoRegistrado:
        """tipo/marca/modelo: equipo; problema_reportado: falla; client_id: dueño; numero_serie: opcional."""

        entrada = self._parse(EquipoCrear, {
            'tipo': tipo, 'marca': marca, 'modelo': modelo,
            'problema_reportado': problema_reportado, 'client_id': client_id,
            'numero_serie': numero_serie or None,
        }, entrada=True)
        data = await self.api.request('POST', '/equipos/', json=entrada.model_dump())

        return self._parse(EquipoRegistrado, data)

```

### `frontend/src/services/users_service.py`

Rutas de administración de usuarios, independientes de las otras entidades.

SHA-256: `8056491a4063338041bb4eacfe861cefd22be891470214be18ca6b0ab8611658`.

- L10: `listar(self)`: Retorna el personal registrado.
- L14: `crear(self, username, password, role)`: username/password: credenciales; role: admin, tecnico o recepcion.
- L24: `cambiar_rol(self, user_id, role)`: user_id: cuenta destino; role: nuevo rol de dominio.
- L36: `activar(self, user_id, activo=True)`: user_id: cuenta destino; activo: habilitar o inhabilitar acceso.

**Código de referencia**

```python
"""Rutas de administración de usuarios, independientes de las otras entidades."""
from services.api_client import ModuleService
from models.base import Mensaje
from models.users import Usuario, UsuarioCrear, UsuarioRegistrado, RolActualizar


class UsersService(ModuleService):
    """Operaciones de usuarios autorizadas por el backend para admin."""

    async def listar(self) -> list[Usuario]:
        """Retorna el personal registrado."""
        return await self._list('/usuarios/', Usuario)

    async def crear(self, username, password, role) -> UsuarioRegistrado:
        """username/password: credenciales; role: admin, tecnico o recepcion."""

        entrada = self._parse(UsuarioCrear, {
            'username': username, 'password': password, 'role': role,
        }, entrada=True)
        data = await self.api.request('POST', '/usuarios/', json=entrada.model_dump())

        return self._parse(UsuarioRegistrado, data)

    async def cambiar_rol(self, user_id, role) -> Mensaje:
        """user_id: cuenta destino; role: nuevo rol de dominio."""

        entrada = self._parse(RolActualizar, {'role': role}, entrada=True)
        data = await self.api.request(
            'PATCH',
            f'/usuarios/{user_id}/rol',
            json=entrada.model_dump(),
        )

        return self._parse(Mensaje, data)

    async def activar(self, user_id, activo=True) -> Mensaje:
        """user_id: cuenta destino; activo: habilitar o inhabilitar acceso."""

        action = 'habilitar' if activo else 'inhabilitar'
        data = await self.api.request('PATCH', f'/usuarios/{user_id}/{action}')

        return self._parse(Mensaje, data)

```

### `frontend/src/ui/__init__.py`

Marcador de paquete.

SHA-256: `e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855`.



### `frontend/src/ui/branding.py`

Carga el logo compartido por login, navegación e iconos de ventana.

SHA-256: `8d6589ef10ea75bf2735daf11da5690a2bd96c6351eadd6addf2a0e58cfa890a`.

- L8: `load_logo(master, size)`: master: widget Tk propietario; size: ancho y alto máximos en píxeles.
- L18: `set_window_icon(window)`: window: ventana; conserva la imagen para que Tk no elimine el icono.

**Código de referencia**

```python
"""Carga el logo compartido por login, navegación e iconos de ventana."""
from pathlib import Path
from PIL import Image, ImageTk

LOGO_PATH = Path(__file__).resolve().parents[1] / 'assets' / 'img' / 'Logo Fixify.png'


def load_logo(master, size):
    """master: widget Tk propietario; size: ancho y alto máximos en píxeles."""

    with Image.open(LOGO_PATH) as source:
        logo = source.convert('RGBA')
        logo.thumbnail(size, Image.Resampling.LANCZOS)

    return ImageTk.PhotoImage(logo, master=master)


def set_window_icon(window):
    """window: ventana; conserva la imagen para que Tk no elimine el icono."""

    window._fixify_icon = load_logo(window, (64, 64))
    window.iconphoto(False, window._fixify_icon)

```

### `frontend/src/ui/components/__init__.py`

Exporta los componentes comunes; cada uno vive en su propio archivo.

SHA-256: `bfa7f0e50712590e87a2a2c55c58e95a01df244154488cec0218ffa6da0c0f97`.



**Código de referencia**

```python
"""Exporta los componentes comunes; cada uno vive en su propio archivo."""
from .async_frame import AsyncFrame
from .collection_view import CollectionView
from .field import Field
from .form_dialog import FormDialog
from .table import create_table

__all__ = ['AsyncFrame', 'CollectionView', 'Field', 'FormDialog', 'create_table']

```

### `frontend/src/ui/components/async_frame.py`

Base de vista que cancela sus peticiones cuando se destruye.

SHA-256: `f04e8202091c7b74e9e406149c23da7347b6adb7ecaf850cbb9f8d58b6fffdaf`.

- L8: `__init__(self, master, services, runner, role)`: master: contenedor; services: módulos API; runner: puente asyncio; role: rol de dominio.
- L14: `destroy(self)`: Descarta tareas y callbacks antes de destruir los controles.
- L19: `request(self, operation, *args, on_success, on_error=None, **kwargs)`: operation: método async; args/kwargs: sus datos; callbacks normales en Tk.

Vincula la tarea a esta vista y usa _failed si no se indica on_error.
Capturar los valores de widgets antes de llamar; no crear hilos ni loops.
Devuelve el Future solo para quien necesite cancelar una lectura anterior.
- L29: `_failed(self, error)`: error: excepción del service; muestra el mensaje en el estado de la vista.

**Código de referencia**

```python
"""Base de vista que cancela sus peticiones cuando se destruye."""
import ttkbootstrap as ttk


class AsyncFrame(ttk.Frame):
    """Base de vista que cancela sus peticiones cuando se destruye."""

    def __init__(self, master, services, runner, role):
        """master: contenedor; services: módulos API; runner: puente asyncio; role: rol de dominio."""
        super().__init__(master)

        self.services, self.runner, self.role = services, runner, role

    def destroy(self):
        """Descarta tareas y callbacks antes de destruir los controles."""
        self.runner.cancel_owner(self)
        super().destroy()

    def request(self, operation, *args, on_success, on_error=None, **kwargs):
        """operation: método async; args/kwargs: sus datos; callbacks normales en Tk.

        Vincula la tarea a esta vista y usa _failed si no se indica on_error.
        Capturar los valores de widgets antes de llamar; no crear hilos ni loops.
        Devuelve el Future solo para quien necesite cancelar una lectura anterior.
        """
        return self.runner.submit(operation(*args, **kwargs), self,
                                  on_success, on_error or self._failed)

    def _failed(self, error):
        """error: excepción del service; muestra el mensaje en el estado de la vista."""
        self.status.configure(text=str(error))

```

### `frontend/src/ui/components/collection_view.py`

Comparte listado, recarga, alta y estados visuales entre módulos.

SHA-256: `67028697096b309956a63230f7f0badb4abc9d19b8baff858d37c6a35bb6eb0a`.

- L25: `__init__(self, master, services, runner, role)`: Recibe dependencias comunes; cada subclase define columnas y operaciones.
- L58: `fetch(self)`: Consulta listar() del servicio declarado; sobrescribir para filtros propios.
- L62: `save(self, values)`: values: campos; delega crear(**values), salvo conversiones de la subclase.
- L67: `service(self)`: Resuelve service_name en el contenedor compartido, sin crear conexiones.
- L75: `row(self, record)`: record: entidad API; la subclase la convierte a valores de tabla.
- L79: `load(self)`: Consulta sin bloquear; asyncio no detecta cambios remotos automáticamente.
- L90: `_loaded(self, records)`: records: lista validada; reemplaza filas y actualiza estado.
- L104: `_failed(self, error)`: error: fallo visible; conserva datos anteriores y habilita reintento.
- L109: `open_form(self)`: Abre el formulario común con los campos del módulo.
- L125: `_created(self, result: Mensaje)`: result: confirmación de alta; informa y vuelve a consultar registros.

**Código de referencia**

```python
"""Comparte listado, recarga, alta y estados visuales entre módulos."""
from tkinter import messagebox
import ttkbootstrap as ttk
from models.base import Mensaje
from models.equipment import EquipoRegistrado
from .async_frame import AsyncFrame
from .form_dialog import FormDialog
from .table import create_table


class CollectionView(AsyncFrame):
    """Base de listado/alta: configurar title, columns y fields en la subclase.

    service_name conecta el módulo a listar/crear sin gestionar el runner.
    fetch() retorna una lista de modelos; save(values) retorna una confirmación;
    row(record) devuelve una tupla en el mismo orden que columns. fetch/save
    son async y no acceden a Tk. El constructor carga datos automáticamente:
    preparar cualquier dependencia adicional antes de llamar a super().__init__.
    """
    title = ''
    columns = ()
    fields = ()
    service_name = None

    def __init__(self, master, services, runner, role):
        """Recibe dependencias comunes; cada subclase define columnas y operaciones."""
        super().__init__(master, services, runner, role)

        self.records = []
        self._load_future = None
        self._dialog = None
        ttk.Label(
            self,
            text=self.title,
            font=('Arial', 18, 'bold'),
        ).pack(anchor='w', pady=(0, 15))

        bar = ttk.Frame(self)
        bar.pack(fill='x', pady=(0, 10))

        self.refresh = ttk.Button(bar, text='Actualizar', command=self.load)
        self.refresh.pack(side='left')

        self.create = ttk.Button(
            bar,
            text='+ Nuevo',
            bootstyle='success',
            command=self.open_form,
        )
        self.create.pack(side='right')

        self.status = ttk.Label(self, text='', wraplength=750)
        self.status.pack(fill='x', pady=(0, 10))

        self.table = create_table(self, self.columns)
        self.load()

    async def fetch(self):
        """Consulta listar() del servicio declarado; sobrescribir para filtros propios."""
        return await self.service.listar()

    async def save(self, values):
        """values: campos; delega crear(**values), salvo conversiones de la subclase."""
        return await self.service.crear(**values)

    @property
    def service(self):
        """Resuelve service_name en el contenedor compartido, sin crear conexiones."""

        if not self.service_name:
            raise ValueError('La vista debe declarar service_name o sobrescribir fetch/save.')

        return getattr(self.services, self.service_name)

    def row(self, record):
        """record: entidad API; la subclase la convierte a valores de tabla."""
        raise NotImplementedError

    def load(self):
        """Consulta sin bloquear; asyncio no detecta cambios remotos automáticamente."""

        if self._load_future is not None:
            self._load_future.cancel()

        self.refresh.configure(state='disabled')
        self.status.configure(text='Cargando…')

        self._load_future = self.request(self.fetch, on_success=self._loaded)

    def _loaded(self, records):
        """records: lista validada; reemplaza filas y actualiza estado."""

        self.records = records

        for item in self.table.get_children():
            self.table.delete(item)

        for index, record in enumerate(records):
            self.table.insert('', 'end', iid=str(index), values=self.row(record))

        self.status.configure(text=f'{len(records)} registros')
        self.refresh.configure(state='normal')

    def _failed(self, error):
        """error: fallo visible; conserva datos anteriores y habilita reintento."""
        self.status.configure(text=str(error))
        self.refresh.configure(state='normal')

    def open_form(self):
        """Abre el formulario común con los campos del módulo."""

        if self._dialog is not None and self._dialog.winfo_exists():
            self._dialog.lift()
            return

        self._dialog = FormDialog(
            self,
            self.title,
            self.fields,
            self.runner,
            self.save,
            self._created,
        )

    def _created(self, result: Mensaje):
        """result: confirmación de alta; informa y vuelve a consultar registros."""

        folio = result.folio if isinstance(result, EquipoRegistrado) else None
        messagebox.showinfo('Registro guardado', f'Guardado correctamente. Folio: {folio}'
                            if folio else 'Guardado correctamente.', parent=self)
        # load cancela solo la lectura previa; no interrumpe otras escrituras.
        self.load()

```

### `frontend/src/ui/components/field.py`

name: clave API; label: texto; required: obligatorio; secret: contraseña; choices: opciones.

SHA-256: `f456a4144d364211b053edaad04bc6423b0afc24a597f7f41dde3e4d8927a043`.



**Código de referencia**

```python
"""name: clave API; label: texto; required: obligatorio; secret: contraseña; choices: opciones."""
from dataclasses import dataclass


@dataclass(frozen=True)
class Field:
    """name: clave API; label: texto; required: obligatorio; secret: contraseña; choices: opciones."""
    name: str
    label: str
    required: bool = True
    secret: bool = False
    choices: tuple = ()

```

### `frontend/src/ui/components/form_dialog.py`

Formulario genérico con validación de campos requeridos y guardado async.

SHA-256: `2fbaaf7d88a639f0948dfee4e409150f95c8b64ad91d2ba98f52a74a64563ac7`.

- L8: `__init__(self, master, title, fields, runner, save, on_saved)`: master: vista; title: título; fields: campos; runner: tareas; save/on_saved: guardar/confirmar.
- L54: `_submit(self)`: Lee controles en Tk; conserva espacios de contraseñas y evita doble envío.
- L82: `_saved(self, result)`: result: respuesta API; notifica al módulo y cierra el formulario.
- L87: `_failed(self, error)`: error: fallo de solicitud; permite corregir y reintentar.
- L92: `destroy(self)`: Cancela el guardado pendiente y evita callbacks sobre un modal cerrado.

**Código de referencia**

```python
"""Formulario genérico con validación de campos requeridos y guardado async."""
import ttkbootstrap as ttk


class FormDialog(ttk.Toplevel):
    """Formulario genérico con validación de campos requeridos y guardado async."""

    def __init__(self, master, title, fields, runner, save, on_saved):
        """master: vista; title: título; fields: campos; runner: tareas; save/on_saved: guardar/confirmar."""
        super().__init__(master)

        self.runner, self.save, self.on_saved = runner, save, on_saved
        self.fields, self.inputs = fields, {}
        self.title(title)
        self.resizable(False, False)

        body = ttk.Frame(self, padding=20)
        body.pack(fill='both', expand=True)
        ttk.Label(body, text=title, font=('Arial', 14, 'bold')).pack(pady=(0, 15))

        for field in fields:
            ttk.Label(body, text=field.label).pack(anchor='w', pady=(5, 0))

            if field.choices:
                widget = ttk.Combobox(
                    body,
                    values=field.choices,
                    state='readonly',
                    width=38,
                )
                widget.current(0)
            else:
                widget = ttk.Entry(body, width=40, show='*' if field.secret else '')

            widget.pack(fill='x', pady=5)

            self.inputs[field.name] = widget

        self.status = ttk.Label(body, text='', wraplength=350)
        self.status.pack(fill='x', pady=5)

        self.button = ttk.Button(
            body,
            text='Guardar',
            bootstyle='success',
            command=self._submit,
        )
        self.button.pack(pady=(10, 0))
        self.transient(master.winfo_toplevel())
        self.protocol('WM_DELETE_WINDOW', self.destroy)
        self.update_idletasks()
        self.place_window_center()

    def _submit(self):
        """Lee controles en Tk; conserva espacios de contraseñas y evita doble envío."""

        # Las claves Field.name deben coincidir con los argumentos del service.
        # Entry entrega texto: convertir IDs explícitamente antes del service;
        # Contrato usa tipos estrictos y no convierte strings a enteros.
        values = {
            field.name: (
                self.inputs[field.name].get()
                if field.secret
                else self.inputs[field.name].get().strip()
            )
            for field in self.fields
        }
        missing = [
            field.label
            for field in self.fields
            if field.required and not values[field.name]
        ]

        if missing:
            self.status.configure(text='Complete: ' + ', '.join(missing))
            return

        self.button.configure(state='disabled')
        self.status.configure(text='Guardando…')
        self.runner.submit(self.save(values), self, self._saved, self._failed)

    def _saved(self, result):
        """result: respuesta API; notifica al módulo y cierra el formulario."""
        self.on_saved(result)
        self.destroy()

    def _failed(self, error):
        """error: fallo de solicitud; permite corregir y reintentar."""
        self.status.configure(text=str(error))
        self.button.configure(state='normal')

    def destroy(self):
        """Cancela el guardado pendiente y evita callbacks sobre un modal cerrado."""
        self.runner.cancel_owner(self)
        super().destroy()

```

### `frontend/src/ui/components/table.py`

parent: contenedor; columns: encabezados; height: filas visibles; retorna Treeview.

SHA-256: `a635881ba12407d7c72a47a9adf5b6306cec732251911497eb71fac104f68102`.

- L5: `create_table(parent, columns, height=12)`: parent: contenedor; columns: encabezados; height: filas visibles; retorna Treeview.

**Código de referencia**

```python
"""parent: contenedor; columns: encabezados; height: filas visibles; retorna Treeview."""
import ttkbootstrap as ttk


def create_table(parent, columns, height=12):
    """parent: contenedor; columns: encabezados; height: filas visibles; retorna Treeview."""

    frame = ttk.Frame(parent)
    frame.pack(fill='both', expand=True)

    table = ttk.Treeview(frame, columns=columns, show='headings', height=height)

    for column in columns:
        table.heading(column, text=column)
        table.column(column, anchor='center', width=130)

    scroll = ttk.Scrollbar(frame, orient='vertical', command=table.yview)
    table.configure(yscrollcommand=scroll.set)
    table.pack(side='left', fill='both', expand=True)
    scroll.pack(side='right', fill='y')
    return table

```

### `frontend/src/ui/login.py`

Login de escritorio con solicitudes asíncronas y sesión en memoria.

SHA-256: `548876c169f6e2f23140f0a7f6b5e741d4aaabc4010313470c46451981d9a636`.

- L11: `__init__(self, services=None)`: services: contenedor opcional para inyectar una API de pruebas.
- L25: `crear_widgets(self)`: Construye campos con etiquetas estables y feedback de conexión.
- L56: `verificar_login(self)`: Valida presencia; preserva contraseña y evita solicitudes repetidas.
- L74: `_logged_in(self, username, data)`: username: cuenta ingresada; data: respuesta validada con token y rol.
- L84: `_failed(self, error)`: error: fallo de login; deja corregir credenciales o reintentar.
- L89: `mostrar_recuperacion(self)`: Informa el procedimiento actual; no existe recuperación automática.
- L98: `cerrar_aplicacion(self)`: Cancela tareas y cierra HTTP antes de destruir la ventana principal.

**Código de referencia**

```python
"""Login de escritorio con solicitudes asíncronas y sesión en memoria."""
import ttkbootstrap as ttk
from app.async_runner import AsyncRunner
from services import Services
from ui.branding import load_logo, set_window_icon


class VentanaLogin(ttk.Window):
    """Ventana principal; administra recursos y el ciclo de vida de la aplicación."""

    def __init__(self, services=None):
        """services: contenedor opcional para inyectar una API de pruebas."""
        super().__init__(title='Fixify — Iniciar sesión', themename='litera', size=(450, 460))
        self.geometry('450x540')
        set_window_icon(self)

        self.services = services if services is not None else Services()
        self.runner = AsyncRunner(self)
        self._closing = False
        self.resizable(False, False)
        self.protocol('WM_DELETE_WINDOW', self.cerrar_aplicacion)
        self.crear_widgets()
        self.place_window_center()

    def crear_widgets(self):
        """Construye campos con etiquetas estables y feedback de conexión."""

        body = ttk.Frame(self, padding=35)
        body.pack(fill='both', expand=True)

        # Tk requiere una referencia a la imagen mientras el widget esté visible.
        self.logo = load_logo(self, (140, 140))
        ttk.Label(body, image=self.logo).pack(pady=(0, 20))
        ttk.Label(body, text='Usuario').pack(anchor='w')

        self.entry_usuario = ttk.Entry(body)
        self.entry_usuario.pack(fill='x', pady=(5, 15))
        ttk.Label(body, text='Contraseña').pack(anchor='w')

        self.entry_password = ttk.Entry(body, show='*')
        self.entry_password.pack(fill='x', pady=5)

        self.status = ttk.Label(body, text='', wraplength=370)
        self.status.pack(fill='x', pady=10)

        self.button = ttk.Button(
            body,
            text='Iniciar sesión',
            command=self.verificar_login,
        )
        self.button.pack(fill='x', pady=10)
        ttk.Button(body, text='¿Olvidaste tu contraseña?', bootstyle='link',
                   command=self.mostrar_recuperacion).pack()
        self.bind('<Return>', lambda event: self.verificar_login())

    def verificar_login(self):
        """Valida presencia; preserva contraseña y evita solicitudes repetidas."""

        if self._closing or str(self.button['state']) == 'disabled':
            return

        username = self.entry_usuario.get().strip()
        password = self.entry_password.get()

        if not username or not password:
            self.status.configure(text='Complete usuario y contraseña.')
            return

        self.button.configure(state='disabled')
        self.status.configure(text='Conectando…')
        self.runner.submit(self.services.auth.login(username, password), self,
                           lambda data: self._logged_in(username, data), self._failed)

    def _logged_in(self, username, data):
        """username: cuenta ingresada; data: respuesta validada con token y rol."""
        from ui.sistema_gestion import SistemaGestion
        self.services.api.set_token(data.access_token)
        self.entry_password.delete(0, 'end')
        self.status.configure(text='')
        self.button.configure(state='normal')
        self.withdraw()
        SistemaGestion(self, username, data.role, self.services, self.runner)

    def _failed(self, error):
        """error: fallo de login; deja corregir credenciales o reintentar."""
        self.status.configure(text=str(error))
        self.button.configure(state='normal')

    def mostrar_recuperacion(self):
        """Informa el procedimiento actual; no existe recuperación automática."""
        from tkinter import messagebox
        messagebox.showinfo(
            'Recuperar cuenta',
            'Contacte al administrador para restablecer su cuenta.',
            parent=self,
        )

    def cerrar_aplicacion(self):
        """Cancela tareas y cierra HTTP antes de destruir la ventana principal."""

        if self._closing:
            return

        self._closing = True
        self.services.api.set_token(None)
        self.button.configure(state='disabled')
        self.status.configure(text='Cerrando…')
        self.runner.close(self.services.api.aclose(), self.destroy)

```

### `frontend/src/ui/sistema_gestion.py`

Ventana de navegación; delega cada módulo a su propia vista.

SHA-256: `6a0611302f8386dbaa271e7afe593e1134cc126a81101c9a5180c0a0e7df99c4`.

- L14: `__init__(self, master, usuario_actual, rol_actual, services, runner)`: master: login; usuario_actual: cuenta; rol_actual: rol API; services/runner: dependencias.
- L55: `show_view(self, view_class)`: view_class: módulo a mostrar; destruye la vista anterior y sus modales.
- L64: `cerrar_sesion(self)`: Borra el JWT local y vuelve a login; no revoca tokens en backend.
- L76: `destroy(self)`: Cancela tareas de la vista antes de cerrar la ventana.

**Código de referencia**

```python
"""Ventana de navegación; delega cada módulo a su propia vista."""
from tkinter import messagebox
import ttkbootstrap as ttk
from ui.views.dashboard import DashboardView
from ui.views.discovery import available_views
from ui.branding import load_logo, set_window_icon

ROLE_LABELS = {'admin': 'Administrador', 'recepcion': 'Recepción', 'tecnico': 'Técnico'}


class SistemaGestion(ttk.Toplevel):
    """Compone navegación por rol con servicios y runner compartidos."""

    def __init__(self, master, usuario_actual, rol_actual, services, runner):
        """master: login; usuario_actual: cuenta; rol_actual: rol API; services/runner: dependencias."""
        super().__init__(master)
        set_window_icon(self)

        self.services, self.runner, self.role = services, runner, rol_actual
        self.view = None
        self.title('Fixify — Gestión de Equipos Tecnológicos')
        self.geometry('1100x700')
        self.minsize(900, 600)
        self.protocol('WM_DELETE_WINDOW', master.cerrar_aplicacion)
        self.columnconfigure(1, weight=1)
        self.rowconfigure(0, weight=1)

        sidebar = ttk.Frame(self, padding=15, bootstyle='secondary')
        sidebar.grid(row=0, column=0, sticky='nsew')

        self.logo = load_logo(self, (150, 150))
        ttk.Label(sidebar, image=self.logo, bootstyle='inverse-secondary').pack(pady=15)
        ttk.Label(sidebar, text=f'Usuario: {usuario_actual}\nRol: {ROLE_LABELS.get(
            self.role,
            self.role,
        )}',
                  bootstyle='inverse-secondary').pack(pady=(0, 20))

        # Cada vista declara etiqueta, roles y orden. El backend debe validar
        # permisos: ocultar un botón solo controla la presentación.
        views = available_views(self.role)

        for label, view_class in views:
            ttk.Button(sidebar, text=label, bootstyle='secondary',
                       command=lambda cls=view_class: self.show_view(cls)).pack(fill='x', pady=5)

        ttk.Button(sidebar, text='Cerrar sesión', bootstyle='danger',
                   command=self.cerrar_sesion).pack(side='bottom', fill='x', pady=10)

        self.content = ttk.Frame(self, padding=25)
        self.content.grid(row=0, column=1, sticky='nsew')
        self.place_window_center()
        self.show_view(DashboardView)

    def show_view(self, view_class):
        """view_class: módulo a mostrar; destruye la vista anterior y sus modales."""

        if self.view is not None:
            self.view.destroy()

        self.view = view_class(self.content, self.services, self.runner, self.role)
        self.view.pack(fill='both', expand=True)

    def cerrar_sesion(self):
        """Borra el JWT local y vuelve a login; no revoca tokens en backend."""

        if messagebox.askyesno(
            'Cerrar sesión',
            '¿Desea salir del sistema?',
            parent=self,
        ):
            self.services.api.set_token(None)
            self.destroy()
            self.master.deiconify()

    def destroy(self):
        """Cancela tareas de la vista antes de cerrar la ventana."""

        if self.view is not None:
            self.view.destroy()

            self.view = None

        super().destroy()

```

### `frontend/src/ui/views/__init__.py`

Vistas separadas por módulo de negocio.

SHA-256: `c4c9f867f92c82bcb95af87e45acdb69c74d365bff62b3fb9992717f7b525628`.



**Código de referencia**

```python
"""Vistas separadas por módulo de negocio."""

```

### `frontend/src/ui/views/clients.py`

Formulario y tabla del módulo de clientes.

SHA-256: `26754b4b6d9d978d5328d9679967d24f68997f29c8340e30ea8067cef53760e5`.

- L20: `fetch(self)`: Consulta clientes desde su service.
- L24: `save(self, values)`: values: nombre, teléfono y datos opcionales capturados.
- L28: `row(self, record: Cliente)`: record: cliente API; convierte campos opcionales vacíos a guion.

**Código de referencia**

```python
"""Formulario y tabla del módulo de clientes."""
from ui.components import CollectionView, Field
from models.clients import Cliente


class ClientsView(CollectionView):
    """Configura clientes reutilizando la vista común de colección."""
    # Plantilla de módulo: campos -> save -> service -> contrato de respuesta.
    # row solo adapta presentación; no realiza consultas ni valida la API.
    title = 'Directorio de Clientes'
    menu_label = 'Clientes'
    allowed_roles = ('admin', 'recepcion')
    menu_order = 20
    service_name = 'clients'
    columns = ('ID', 'Nombre', 'Teléfono', 'Correo', 'Dirección')
    fields = (Field('nombre', 'Nombre completo'), Field('telefono', 'Teléfono'),
              Field('correo', 'Correo (opcional)', required=False),
              Field('direccion', 'Dirección (opcional)', required=False))

    async def fetch(self):
        """Consulta clientes desde su service."""
        return await self.services.clients.listar()

    async def save(self, values):
        """values: nombre, teléfono y datos opcionales capturados."""
        return await self.services.clients.crear(**values)

    def row(self, record: Cliente):
        """record: cliente API; convierte campos opcionales vacíos a guion."""
        return (record.id, record.nombre, record.telefono, record.correo or '-', record.direccion or '-')

```

### `frontend/src/ui/views/dashboard.py`

Dashboard con consultas concurrentes y métricas limitadas por rol.

SHA-256: `acc4c8ce4bd150fd5325858cce5d8c801cbe0245cbb3094da0ce7932ca6e680a`.

- L13: `__init__(self, master, services, runner, role)`: Recibe dependencias comunes y solicita solo colecciones autorizadas.
- L61: `_fetch(self)`: Consulta métricas en paralelo; conserva errores independientes.
- L68: `_loaded(self, results)`: results: una lista o error por métrica; muestra fallos sin ocultarlos.
- L98: `_failed(self, error)`: error: fallo general del dashboard; mantiene la ventana utilizable.

**Código de referencia**

```python
"""Dashboard con consultas concurrentes y métricas limitadas por rol."""
import asyncio
import ttkbootstrap as ttk
from ui.components import AsyncFrame, create_table


class DashboardView(AsyncFrame):
    """Muestra conteos sin confundir fallos de red con cero registros."""
    menu_label = 'Dashboard'
    allowed_roles = ('admin', 'recepcion', 'tecnico')
    menu_order = 0

    def __init__(self, master, services, runner, role):
        """Recibe dependencias comunes y solicita solo colecciones autorizadas."""
        super().__init__(master, services, runner, role)
        ttk.Label(
            self,
            text='Dashboard General',
            font=('Arial', 18, 'bold'),
        ).pack(anchor='w', pady=(0, 20))

        self.status = ttk.Label(self, text='Cargando…', wraplength=750)
        self.status.pack(fill='x', pady=10)

        cards = ttk.Frame(self)
        cards.pack(fill='x', pady=10)

        self.metrics = {}
        self.queries = []

        if role in ('admin', 'recepcion'):
            self.queries = [
                ('Equipos', services.equipment.listar),
                ('Clientes', services.clients.listar),
            ]

        if role == 'admin':
            self.queries.append(('Usuarios', services.users.listar))

        for index, (label, _) in enumerate(self.queries):
            card = ttk.Labelframe(cards, text=label, padding=15)
            card.grid(row=0, column=index, sticky='nsew', padx=5)
            cards.columnconfigure(index, weight=1)

            metric = ttk.Label(card, text='…', font=('Arial', 22, 'bold'))
            metric.pack()

            self.metrics[label] = metric

        if not self.queries:
            self.status.configure(text='Sesión de técnico activa. El módulo de órdenes asignadas está pendiente.')
            return

        self.table = create_table(
            self,
            ('Folio', 'Tipo', 'Marca', 'Modelo', 'Problema', 'ID Cliente'),
            height=8,
        )
        self.request(self._fetch, on_success=self._loaded)

    async def _fetch(self):
        """Consulta métricas en paralelo; conserva errores independientes."""
        return await asyncio.gather(
            *(fetch() for _, fetch in self.queries),
            return_exceptions=True,
        )

    def _loaded(self, results):
        """results: una lista o error por métrica; muestra fallos sin ocultarlos."""

        errors = []

        for (label, _), result in zip(self.queries, results):
            if isinstance(result, Exception):
                self.metrics[label].configure(text='No disponible')
                errors.append(f'{label}: {result}')
                continue

            self.metrics[label].configure(text=str(len(result)))

            if label == 'Equipos':
                for equipment in sorted(result, key=lambda eq: eq.id, reverse=True)[:8]:
                    self.table.insert(
                        '',
                        'end',
                        values=(
                            f"EQ-{equipment.id:03d}",
                            equipment.tipo,
                            equipment.marca,
                            equipment.modelo,
                            equipment.problema_reportado,
                            equipment.client_id,
                        ),
                    )

        self.status.configure(text='\n'.join(errors) or 'Datos actualizados. Equipos ordenados por ID descendente.')

    def _failed(self, error):
        """error: fallo general del dashboard; mantiene la ventana utilizable."""
        self.status.configure(text=str(error))

```

### `frontend/src/ui/views/discovery.py`

Encuentra vistas locales y prepara las entradas del menú según el rol.

Una vista hereda AsyncFrame y declara menu_label, allowed_roles y,
opcionalmente, menu_order. No crea widgets ni instancias de las vistas.
Los módulos se importan: evitar efectos secundarios al nivel del archivo.
Este filtro controla presentación; la API debe autorizar cada operación.

SHA-256: `417f7b9940a85f5a59f7baffaa44b77b2bfb0e7cfe1a1e6693e93bb8a25d3540`.

- L14: `available_views(role)`: role: código API como admin, recepcion o tecnico.

Retorna lista de (etiqueta, clase), ordenada por menu_order y etiqueta.
Sin etiqueta o sin el rol en allowed_roles, la vista no se muestra.
menu_order vale 100 si se omite. Los errores de importación se propagan.

**Código de referencia**

```python
"""Encuentra vistas locales y prepara las entradas del menú según el rol.

Una vista hereda AsyncFrame y declara menu_label, allowed_roles y,
opcionalmente, menu_order. No crea widgets ni instancias de las vistas.
Los módulos se importan: evitar efectos secundarios al nivel del archivo.
Este filtro controla presentación; la API debe autorizar cada operación.
"""
from importlib import import_module
from pkgutil import iter_modules

from ui.components import AsyncFrame


def available_views(role):
    """role: código API como admin, recepcion o tecnico.

    Retorna lista de (etiqueta, clase), ordenada por menu_order y etiqueta.
    Sin etiqueta o sin el rol en allowed_roles, la vista no se muestra.
    menu_order vale 100 si se omite. Los errores de importación se propagan.
    """
    import ui.views

    views = []

    for entry in iter_modules(ui.views.__path__):
        if entry.name.startswith('_') or entry.name == 'discovery':
            continue

        # Ignora módulos internos y este descubridor para evitar autorregistro.
        module = import_module(f'ui.views.{entry.name}')

        for cls in vars(module).values():
            if (
                isinstance(cls, type)
                and issubclass(cls, AsyncFrame)
                and cls.__module__ == module.__name__
                and getattr(cls, 'menu_label', None)
                and role in getattr(cls, 'allowed_roles', ())
            ):
                views.append(cls)

    views.sort(key=lambda cls: (getattr(cls, 'menu_order', 100), cls.menu_label))
    return [(cls.menu_label, cls) for cls in views]

```

### `frontend/src/ui/views/equipment.py`

Vista de equipos con selección asíncrona de clientes existentes.

SHA-256: `be7833ddbc2f70da179df0d4ac9e4887b39d2eec2c3541912a8a2d9be2032a00`.

- L16: `fetch(self)`: Consulta los equipos disponibles mediante su service.
- L20: `row(self, record: Equipo)`: record: equipo API; formatea su identificador visual.
- L25: `open_form(self)`: Carga propietarios sin detener Tk; evita consultas duplicadas.
- L37: `_clients_failed(self, error)`: error: fallo de consulta; restaura el botón de alta.
- L42: `_clients_ready(self, clients: list[Cliente])`: clients: lista API; crea etiquetas legibles asociadas a IDs.
- L59: `save(values)`: values: formulario; traduce la etiqueta elegida a ID de cliente.

**Código de referencia**

```python
"""Vista de equipos con selección asíncrona de clientes existentes."""
from ui.components import CollectionView, Field, FormDialog
from models.equipment import Equipo
from models.clients import Cliente


class EquipmentView(CollectionView):
    """Reutiliza la tabla y prepara el selector de propietario antes del alta."""
    menu_label = 'Equipos'
    allowed_roles = ('admin', 'recepcion')
    menu_order = 10
    service_name = 'equipment'
    title = 'Gestión de Equipos'
    columns = ('Folio', 'Tipo', 'Marca', 'Modelo', 'Nº Serie', 'Problema', 'ID Cliente')

    async def fetch(self):
        """Consulta los equipos disponibles mediante su service."""
        return await self.services.equipment.listar()

    def row(self, record: Equipo):
        """record: equipo API; formatea su identificador visual."""
        return (f"EQ-{record.id:03d}", record.tipo, record.marca, record.modelo,
                record.numero_serie or '-', record.problema_reportado, record.client_id)

    def open_form(self):
        """Carga propietarios sin detener Tk; evita consultas duplicadas."""

        if self._dialog is not None and self._dialog.winfo_exists():
            self._dialog.lift()
            return

        self.create.configure(state='disabled')
        self.status.configure(text='Cargando clientes…')
        self.request(self.services.clients.listar, on_success=self._clients_ready,
                     on_error=self._clients_failed)

    def _clients_failed(self, error):
        """error: fallo de consulta; restaura el botón de alta."""
        self.create.configure(state='normal')
        self.status.configure(text=str(error))

    def _clients_ready(self, clients: list[Cliente]):
        """clients: lista API; crea etiquetas legibles asociadas a IDs."""
        self.create.configure(state='normal')

        if not clients:
            self.status.configure(text='Registre al menos un cliente antes de ingresar un equipo.')
            return

        owners = {f"[{c.id}] {c.nombre} ({c.telefono})": c.id for c in clients}
        fields = (Field('client_id', 'Cliente', choices=tuple(owners)),
                  Field('tipo', 'Tipo de equipo'), Field('marca', 'Marca'),
                  Field(
                      'modelo',
                      'Modelo',
                  ), Field('numero_serie', 'Serie (opcional)', required=False),
                  Field('problema_reportado', 'Problema reportado'))

        async def save(values):
            """values: formulario; traduce la etiqueta elegida a ID de cliente."""
            return await self.services.equipment.crear(**{**values, 'client_id': owners[values['client_id']]})

        self._dialog = FormDialog(
            self,
            'Registrar Equipo',
            fields,
            self.runner,
            save,
            self._created,
        )
        self.status.configure(text='Seleccione el cliente propietario en el formulario.')

```

### `frontend/src/ui/views/users.py`

Vista de administración de cuentas y roles.

SHA-256: `d4547bbf0ff5b58ca745239b1c3fb3a9ce785c07444a9d25f5902ddb7ebd10e3`.

- L24: `__init__(self, master, services, runner, role)`: Recibe dependencias comunes y agrega acciones administrativas.
- L50: `fetch(self)`: Consulta personal autorizado para administración.
- L54: `save(self, values)`: values: credenciales y etiqueta de rol; convierte esta al valor API.
- L58: `row(self, record: Usuario)`: record: usuario API; muestra estado legible sin datos de contraseña.
- L62: `_selected(self)`: Retorna el registro seleccionado o informa que falta selección.
- L73: `change_role(self)`: Solicita cambio al rol seleccionado para la cuenta elegida.
- L89: `toggle_active(self)`: Alterna acceso sin eliminar el historial del usuario.
- L101: `_update(self, operation, *args)`: operation: método async; args: datos; bloquea acciones repetidas.
- L118: `_updated(self, result)`: result: respuesta API; restaura acciones y recarga cuentas.
- L126: `_update_failed(self, error)`: error: fallo; permite reintentar sin ocultar la causa.

**Código de referencia**

```python
"""Vista de administración de cuentas y roles."""
from tkinter import messagebox
import ttkbootstrap as ttk
from ui.components import CollectionView, Field
from models.users import Usuario

ROLES = {'Administrador': 'admin', 'Técnico': 'tecnico', 'Recepción': 'recepcion'}


class UsersView(CollectionView):
    """Permite altas y cambios de acceso sobre el usuario seleccionado."""
    menu_label = 'Usuarios'
    allowed_roles = ('admin',)
    menu_order = 30
    service_name = 'users'
    title = 'Administración de Usuarios'
    columns = ('ID', 'Usuario', 'Rol', 'Estado')
    fields = (
        Field('username', 'Usuario (mín. 4 caracteres)'),
        Field('password', 'Contraseña (mín. 6 caracteres)', secret=True),
        Field('role', 'Rol', choices=tuple(ROLES)),
    )

    def __init__(self, master, services, runner, role):
        """Recibe dependencias comunes y agrega acciones administrativas."""
        super().__init__(master, services, runner, role)

        actions = ttk.Frame(self)
        actions.pack(fill='x', pady=10)

        self.role_input = ttk.Combobox(
            actions,
            values=tuple(ROLES),
            state='readonly',
            width=18,
        )
        self.role_input.set('Técnico')
        self.role_input.pack(side='left', padx=(0, 10))

        self.action_buttons = []

        for label, callback in (
            ('Cambiar rol', self.change_role),
            ('Activar / Desactivar', self.toggle_active),
        ):
            button = ttk.Button(actions, text=label, command=callback)
            button.pack(side='left', padx=5)
            self.action_buttons.append(button)

    async def fetch(self):
        """Consulta personal autorizado para administración."""
        return await self.services.users.listar()

    async def save(self, values):
        """values: credenciales y etiqueta de rol; convierte esta al valor API."""
        return await self.services.users.crear(**{**values, 'role': ROLES[values['role']]})

    def row(self, record: Usuario):
        """record: usuario API; muestra estado legible sin datos de contraseña."""
        return (record.id, record.username, record.role, 'Activo' if record.is_active else 'Inactivo')

    def _selected(self):
        """Retorna el registro seleccionado o informa que falta selección."""

        selected = self.table.selection()

        if not selected:
            self.status.configure(text='Seleccione un usuario en la tabla.')
            return None

        return self.records[int(selected[0])]

    def change_role(self):
        """Solicita cambio al rol seleccionado para la cuenta elegida."""

        user = self._selected()

        if user and messagebox.askyesno(
            'Cambiar rol',
            '¿Cambiar el rol de esta cuenta?',
            parent=self,
        ):
            self._update(
                self.services.users.cambiar_rol,
                user.id,
                ROLES[self.role_input.get()],
            )

    def toggle_active(self):
        """Alterna acceso sin eliminar el historial del usuario."""

        user = self._selected()

        if user and messagebox.askyesno(
            'Cambiar acceso',
            '¿Cambiar el acceso de esta cuenta?',
            parent=self,
        ):
            self._update(self.services.users.activar, user.id, not user.is_active)

    def _update(self, operation, *args):
        """operation: método async; args: datos; bloquea acciones repetidas."""

        if self._load_future is not None:
            self._load_future.cancel()

        for button in [self.refresh, self.create, *self.action_buttons]:
            button.configure(state='disabled')

        self.status.configure(text='Actualizando cuenta…')
        self.request(
            operation,
            *args,
            on_success=self._updated,
            on_error=self._update_failed,
        )

    def _updated(self, result):
        """result: respuesta API; restaura acciones y recarga cuentas."""

        for button in [self.create, *self.action_buttons]:
            button.configure(state='normal')

        self.load()

    def _update_failed(self, error):
        """error: fallo; permite reintentar sin ocultar la causa."""

        for button in [self.create, *self.action_buttons]:
            button.configure(state='normal')

        self._failed(error)

```

## 16. Comprobaciones y próximos pasos

Comandos desde la raíz:

```powershell
python -m unittest discover -s frontend/tests -v
python -m unittest discover -s backend/tests -v
python frontend/src/app/main.py
```

Las pruebas de frontend verifican 15 escenarios de contratos, modelos, errores, filas de vistas, concurrencia y ciclo de cierre sin abrir Tk. Las pruebas de equipos verifican 5 escenarios: alta/listado con columnas físicas del diagrama, validación y cliente inexistente, serie opcional/orden estable, permisos y consumo del router real desde EquipmentService con ASGITransport. Utilizan SQLite temporal: comprueban mapeo/contrato, no la conectividad ni todas las particularidades de PostgreSQL.

Pendiente: aceptación visual y ejecución contra esquema real de Supabase. El diagrama define nombres, no evidencia que todas las columnas de los otros módulos están sincronizadas. No ejecutar el SQL destructivo del repositorio para comprobar lectura.

### Referencias

- Código vigente: `frontend/src/` y `backend/src/modules/equipment/`.
- Diagrama proporcionado: `D:/Descargas/Diagramas.pdf`, página 1; referencia de columnas físicas.
- [HTTPX: AsyncClient y cierre](https://www.python-httpx.org/async/).
- [HTTPX: parámetros, cuerpos y respuestas](https://www.python-httpx.org/quickstart/).
- [HTTPX: timeout](https://www.python-httpx.org/advanced/timeouts/).
- [Python: coroutines, gather y coordinación entre hilos](https://docs.python.org/3/library/asyncio-task.html).

La descripción del flujo y sus límites corresponde al código local. Las referencias técnicas explican APIs de bibliotecas; no implican que Fixify implemente todas sus capacidades.

Validación actual: 20 pruebas del frontend aprobadas, incluyendo descubrimiento, permisos y delegación de tareas.

## Actualización: alineación SQL y prueba real en Supabase (02-Oct-2026)

El backend ahora mapea las diez tablas del SQL local, incluidas usuarios/roles. Se renombró equipos.problema_reportado a falla_reportada en Supabase conservando datos; users y la columna adicional direccion se conservan remotamente, pero no forman parte del ORM del SQL base. No hay create_all durante el arranque.

Se creó fixify_prueba (ID 2, recepción) y se verificó login real y serialización de clientes/equipos contra Supabase. Su contraseña está en backend/usuario_prueba.local.json, excluido de Git. Pasan 7 pruebas backend y 15 frontend.

Limitaciones del esquema original: no existe is_active; activar/desactivar retorna 409 hasta acordar una extensión. Dirección no pertenece al SQL: el backend devuelve None y rechaza alta con dirección no vacía. No se migraron cuentas de users automáticamente. Véase [detalle de alineación](ALINEACION_BASE_DATOS.md). Esta actualización sustituye las afirmaciones anteriores de falta de validación remota para login/clientes/equipos; la aceptación visual permanece pendiente.


## Estado persistido de usuarios: contrato vigente

El campo real es `usuarios.is_active`, expuesto como `is_active` en la API y en los contratos del frontend. La referencia anterior a `is_activate` fue corregida tras revisar el error de PostgreSQL. Activar/desactivar persiste el estado; login rechaza cuentas inactivas con 403. Los tokens ya emitidos no se revocan automáticamente.


## Corrección de error 500: nombre del estado de cuenta

El traceback de PostgreSQL confirmó que la columna existente es usuarios.is_active, no is_activate. Se corrigió el mapeo ORM y el SQL local para usar is_active. El contrato del frontend no cambia. Ocho pruebas backend pasan, incluida una regresión que comprueba el nombre físico de la columna. Esta corrección sustituye las referencias anteriores a is_activate; no requiere renombrar la columna en Supabase.


## Verificación de esta actualización

La refactorización conserva los imports de componentes y las cuatro vistas. Las 15 pruebas del frontend pasan. El laboratorio usa MockTransport y datos sintéticos; no abre ventanas ni conecta a Supabase. La verificación visual del logo queda para la ejecución manual de la app.


## Actualización de entrega RF01–RF04 — 05-Oct-2026

Se conserva la documentación anterior. Esta sección describe los cambios vigentes de la entrega.

El login incorpora mostrar contraseña, validación de campos y mensajes de error. La navegación destaca la sección activa y muestra módulos según el rol. El backend comprueba los permisos actuales en cada operación; el menú de una ventana abierta se reconstruye al iniciar sesión nuevamente.

Clientes y Equipos incorporan búsqueda local por cualquier columna, desplazamiento horizontal/vertical, filas alternadas, mensajes de lista vacía, **Editar selección** y **Eliminar selección**. Seleccionar una fila antes de actuar. Editar abre un formulario precargado; guardar conserva su ID y recarga la tabla. Para equipos se carga el catálogo de clientes y se conserva el propietario seleccionado.

Eliminar pide confirmación e informa los conflictos enviados por la API. Un cliente con equipos u órdenes y un equipo con órdenes se conservan. Los usuarios siguen administrándose mediante rol y activación/desactivación; no se eliminan físicamente.

`FormDialog(initial=...)` admite valores iniciales sin cambiar el alta. `CollectionView.editable` habilita las acciones en clientes/equipos. Los servicios `editar(id, **values)` y `eliminar(id)` validan los contratos y usan PUT/DELETE. Las operaciones siguen siendo asíncronas y los fallos permiten reintentar.

Dependencias: `ttkbootstrap==1.20.1` consistente entre raíz y frontend; Linux también necesita `python3-tkinter` (paquete del sistema). `unittest` está incluido en Python. Consultar [guía de pruebas](../Test/Guia_Pruebas_Fixify.ipynb) para ejecutar cada archivo y caso individual.


## Corrección de alineación de tablas — 05-Oct-2026

El componente compartido `frontend/src/ui/components/table.py` alinea cada encabezado igual que sus celdas: ID, folio, ID de cliente y estado centrados; nombres, características, teléfono, correo y rol a la izquierda. Antes los encabezados estaban centrados por defecto y el contenido a la izquierda, lo que hacía difícil identificar la columna de cada valor.

Los anchos se definen por campo con `COLUMN_WIDTHS`: nombres, modelos, correo y problema disponen de más espacio. `stretch=False` evita que Tk reduzca o redistribuya automáticamente las columnas al ajustar la ventana. Si la suma de anchos supera el espacio disponible, usar la barra horizontal para ver las columnas restantes. También se puede arrastrar el borde de un encabezado para ampliar manualmente su columna; los textos que superen ese ancho pueden seguir recortándose.

La corrección se aplica a clientes, equipos, usuarios y la tabla de equipos del dashboard, que usan `create_table`. No cambia el orden ni el contenido de los datos devueltos por la API. No requiere dependencias nuevas.


## Acciones visibles de usuarios y colores — 05-Oct-2026

Las acciones de Usuarios están ahora en `CollectionView.action_bar`, sobre la tabla. Esto evita que la tabla expandida deje los botones inferiores fuera de vista. Seleccionar un usuario y usar **Editar usuario** para modificar su nombre y rol en un formulario precargado; guarda mediante `UsersService.editar` y `PUT /usuarios/{user_id}`. La contraseña y el estado se conservan.

**Desactivar usuario** cambia a **Activar usuario** al seleccionar una cuenta inactiva. La confirmación identifica la cuenta y la acción concreta. Las peticiones se bloquean mientras se actualiza el acceso para evitar doble envío. Los usuarios se desactivan, sin borrarlos físicamente.

La paleta utiliza `warning` (amarillo) para editar, `danger` (rojo) para eliminar/desactivar y `success` (verde) para activar o registrar. Clientes y Equipos también utilizan estos colores en la barra de acciones. Las etiquetas explican la acción sin depender exclusivamente del color. No hay dependencias nuevas.
